In [53]:
# ============================================================
# FIRE SOURCE CLASSIFICATION
# NOTEBOOK 1 — FULL TRAINING PIPELINE
# ============================================================
#
# Classes:
# 0 = Other
# 1 = Industrial Fire
# 2 = Gas Flare
# 3 = Agricultural Fire
# 4 = Mining Activity
# 5 = Wildfire
#
# IMPORTANT:
# These source labels are pseudo-labels created from
# contextual evidence. They are NOT NASA FIRMS source labels.
# ============================================================


# ============================================================
# 1. IMPORTS
# ============================================================

import os
import warnings
import joblib
import numpy as np
import pandas as pd

from tqdm.auto import tqdm

from sklearn.neighbors import BallTree

from sklearn.model_selection import train_test_split

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support
)

from sklearn.utils.class_weight import compute_class_weight

warnings.filterwarnings("ignore")


# ============================================================
# 2. FIND DATASET
# ============================================================

print("=" * 70)
print("SEARCHING FOR INPUT CSV")
print("=" * 70)

csv_files = []

for dirname, _, filenames in os.walk("/kaggle/input"):

    for filename in filenames:

        if filename.lower().endswith(".csv"):

            csv_files.append(
                os.path.join(
                    dirname,
                    filename
                )
            )


if len(csv_files) == 0:

    raise FileNotFoundError(
        "No CSV file found inside /kaggle/input"
    )


print("\nCSV files found:")

for p in csv_files:

    print(p)


# Prefer FIRMS + Dynamic World dataset
preferred = [
    p
    for p in csv_files
    if (
        "firms" in p.lower()
        and
        "dynamicworld" in p.lower()
    )
]


if len(preferred) > 0:

    PATH = preferred[0]

else:

    PATH = csv_files[0]


print("\nUsing:")
print(PATH)


# ============================================================
# 3. LOAD DATA
# ============================================================

df = pd.read_csv(
    PATH,
    low_memory=False
)

print("\n" + "=" * 70)
print("MASTER DATASET")
print("=" * 70)

print("Rows   :", len(df))
print("Columns:", len(df.columns))


# ============================================================
# 4. DATE / TIME
# ============================================================

df["acq_date"] = pd.to_datetime(
    df["acq_date"],
    errors="coerce"
)

df["acq_time"] = (
    df["acq_time"]
    .astype(str)
    .str.replace(
        ".0",
        "",
        regex=False
    )
    .str.zfill(4)
)


df["datetime"] = pd.to_datetime(
    df["acq_date"].dt.strftime(
        "%Y-%m-%d"
    )
    + " "
    + df["acq_time"].str[:2]
    + ":"
    + df["acq_time"].str[2:],
    errors="coerce"
)


df["month"] = (
    df["acq_date"].dt.month
)

df["hour"] = (
    df["datetime"].dt.hour
)

df["minute"] = (
    df["datetime"].dt.minute
)


# Cyclical time
df["hour_sin"] = np.sin(
    2 * np.pi * df["hour"] / 24
)

df["hour_cos"] = np.cos(
    2 * np.pi * df["hour"] / 24
)


print("\nDate range:")
print(
    df["acq_date"].min(),
    "->",
    df["acq_date"].max()
)

print(
    "Missing datetime:",
    df["datetime"].isna().sum()
)


# ============================================================
# 5. NUMERIC FIRMS FEATURES
# ============================================================

for col in [
    "latitude",
    "longitude",
    "brightness",
    "scan",
    "track",
    "bright_t31",
    "frp"
]:

    if col in df.columns:

        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        )


df["frp_log"] = np.log1p(
    df["frp"].clip(
        lower=0
    )
)


# ============================================================
# 6. DYNAMIC WORLD
# ============================================================

dw_cols = [
    "water",
    "trees",
    "grass",
    "flooded_vegetation",
    "crops",
    "shrub_and_scrub",
    "built",
    "bare",
    "snow_and_ice"
]


dw_class_names = {
    0: "water",
    1: "trees",
    2: "grass",
    3: "flooded_vegetation",
    4: "crops",
    5: "shrub_and_scrub",
    6: "built",
    7: "bare",
    8: "snow_and_ice"
}


# ------------------------------------------------------------
# DW availability
# ------------------------------------------------------------

has_dw = (
    df[dw_cols]
    .notna()
    .all(axis=1)
)


df["dw_available"] = (
    has_dw.astype(int)
)


# ------------------------------------------------------------
# DW maximum probability
# ------------------------------------------------------------

df["dw_max_probability"] = (
    df[dw_cols].max(
        axis=1
    )
)


# ------------------------------------------------------------
# DW final label
# ------------------------------------------------------------

df["dw_final_label"] = pd.Series(
    pd.NA,
    index=df.index,
    dtype="Int64"
)


df.loc[
    has_dw,
    "dw_final_label"
] = (
    df.loc[
        has_dw,
        dw_cols
    ]
    .values
    .argmax(
        axis=1
    )
)


df["dw_final_land_cover"] = pd.Series(
    pd.NA,
    index=df.index,
    dtype="string"
)


df.loc[
    has_dw,
    "dw_final_land_cover"
] = (
    df.loc[
        has_dw,
        "dw_final_label"
    ]
    .map(
        dw_class_names
    )
)


print("\n" + "=" * 70)
print("DYNAMIC WORLD")
print("=" * 70)

print(
    "Available:",
    has_dw.sum()
)

print(
    "Unavailable:",
    (~has_dw).sum()
)

print(
    "Coverage:",
    round(
        has_dw.mean() * 100,
        2
    ),
    "%"
)


# ============================================================
# 7. DYNAMIC WORLD DATE DIFFERENCE
# ============================================================

if "dw_date" in df.columns:

    dw_date_clean = (
        df["dw_date"]
        .astype(str)
        .replace(
            "nan",
            np.nan
        )
        .replace(
            "no_data",
            np.nan
        )
    )


    dw_date_clean = pd.to_datetime(
        dw_date_clean,
        errors="coerce"
    )


    df["dw_date_difference_days"] = (
        df["acq_date"]
        -
        dw_date_clean
    ).dt.days

else:

    df["dw_date_difference_days"] = np.nan


# ============================================================
# 8. CONFIDENCE ENCODING
# ============================================================

if "confidence" in df.columns:

    df["confidence_low"] = (
        df["confidence"] == "l"
    ).astype(int)

    df["confidence_nominal"] = (
        df["confidence"] == "n"
    ).astype(int)

    df["confidence_high"] = (
        df["confidence"] == "h"
    ).astype(int)

else:

    df["confidence_low"] = 0
    df["confidence_nominal"] = 0
    df["confidence_high"] = 0


# ============================================================
# 9. PERSISTENCE CALCULATION
# ============================================================

print("\n" + "=" * 70)
print("CALCULATING PERSISTENCE")
print("=" * 70)


df = (
    df
    .sort_values(
        "datetime"
    )
    .reset_index(
        drop=True
    )
)


coords = np.radians(
    df[
        [
            "latitude",
            "longitude"
        ]
    ].values
)


EARTH_RADIUS_KM = 6371.0088

radius_1km = (
    1.0 / EARTH_RADIUS_KM
)


tree_full = BallTree(
    coords,
    metric="haversine"
)


detections_7 = np.zeros(
    len(df),
    dtype=np.int32
)


detections_30 = np.zeros(
    len(df),
    dtype=np.int32
)


night_ratios = np.zeros(
    len(df),
    dtype=np.float32
)


BATCH_SIZE = 5000


for start in tqdm(
    range(
        0,
        len(df),
        BATCH_SIZE
    ),
    desc="Calculating persistence"
):

    end = min(
        start + BATCH_SIZE,
        len(df)
    )


    batch_coords = (
        coords[start:end]
    )


    neighbors = (
        tree_full.query_radius(
            batch_coords,
            r=radius_1km
        )
    )


    for local_i, neighbor_idx in enumerate(
        neighbors
    ):

        i = start + local_i

        target_time = (
            df.iloc[i]["datetime"]
        )


        neighbor_idx = np.asarray(
            neighbor_idx,
            dtype=np.int64
        )


        neighbor_times = (
            df.iloc[
                neighbor_idx
            ]["datetime"]
            .values
        )


        previous_mask = (
            (
                neighbor_times
                <
                np.datetime64(
                    target_time
                )
            )
            &
            (
                neighbor_times
                >=
                (
                    np.datetime64(
                        target_time
                    )
                    -
                    np.timedelta64(
                        30,
                        "D"
                    )
                )
            )
        )


        previous_idx = (
            neighbor_idx[
                previous_mask
            ]
        )


        if len(previous_idx) == 0:

            continue


        previous_times = (
            df.iloc[
                previous_idx
            ]["datetime"]
            .values
        )


        # 30-day
        detections_30[i] = (
            len(previous_idx)
        )


        # 7-day
        seven_day_mask = (
            previous_times
            >=
            (
                np.datetime64(
                    target_time
                )
                -
                np.timedelta64(
                    7,
                    "D"
                )
            )
        )


        detections_7[i] = (
            seven_day_mask.sum()
        )


        # Night ratio
        night_values = (
            df.iloc[
                previous_idx
            ]["daynight"]
            .astype(str)
            .values
        )


        night_count = (
            night_values == "N"
        ).sum()


        night_ratios[i] = (
            night_count
            /
            len(previous_idx)
        )


df["detections_7_days"] = (
    detections_7
)

df["detections_30_days"] = (
    detections_30
)

df["night_ratio"] = (
    night_ratios
)


# ============================================================
# 10. PERSISTENCE SCORE
# ============================================================

p95_7 = (
    df["detections_7_days"]
    .quantile(0.95)
)

p95_30 = (
    df["detections_30_days"]
    .quantile(0.95)
)


if p95_7 <= 0:
    p95_7 = 1.0

if p95_30 <= 0:
    p95_30 = 1.0


score_7 = (
    df["detections_7_days"]
    /
    p95_7
).clip(
    0,
    1
)


score_30 = (
    df["detections_30_days"]
    /
    p95_30
).clip(
    0,
    1
)


df["persistence_score"] = (
    0.6 * score_7
    +
    0.4 * score_30
)


print("\nPersistence statistics:")

print(
    df[
        [
            "detections_7_days",
            "detections_30_days",
            "night_ratio",
            "persistence_score"
        ]
    ].describe()
)


# ============================================================
# 11. OSM LOG FEATURES
# ============================================================

print("\n" + "=" * 70)
print("CREATING OSM LOG FEATURES")
print("=" * 70)


osm_distance_cols = [
    c
    for c in df.columns
    if (
        c.startswith(
            "distance_to_nearest_"
        )
        and
        c.endswith("_m")
    )
]


for col in osm_distance_cols:

    distance_numeric = pd.to_numeric(
        df[col],
        errors="coerce"
    )


    log_col = (
        col[:-2]
        +
        "_log"
    )


    df[log_col] = np.log1p(
        distance_numeric.clip(
            lower=0
        )
    )


print(
    "OSM distance columns:",
    len(osm_distance_cols)
)


# ============================================================
# 12. MINIMUM DISTANCE FEATURES
# ============================================================

def minimum_existing_columns(
    dataframe,
    columns
):

    existing = [
        c
        for c in columns
        if c in dataframe.columns
    ]


    if len(existing) == 0:

        return pd.Series(
            np.nan,
            index=dataframe.index
        )


    return (
        dataframe[
            existing
        ]
        .apply(
            pd.to_numeric,
            errors="coerce"
        )
        .min(
            axis=1
        )
    )


df["min_fossil_distance_m"] = (
    minimum_existing_columns(
        df,
        [
            "distance_to_nearest_flare_m",
            "distance_to_nearest_oil_well_m",
            "distance_to_nearest_gasometer_m"
        ]
    )
)


df["min_mining_distance_m"] = (
    minimum_existing_columns(
        df,
        [
            "distance_to_nearest_mineshaft_m",
            "distance_to_nearest_adit_m",
            "distance_to_nearest_quarry_m"
        ]
    )
)


df["min_agriculture_distance_m"] = (
    minimum_existing_columns(
        df,
        [
            "distance_to_nearest_farmland_m",
            "distance_to_nearest_farmyard_m",
            "distance_to_nearest_orchard_m",
            "distance_to_nearest_vineyard_m",
            "distance_to_nearest_plant_nursery_m",
            "distance_to_nearest_greenhouse_m",
            "distance_to_nearest_allotment_m",
            "distance_to_nearest_agriculture_m"
        ]
    )
)


df["min_wildland_distance_m"] = (
    minimum_existing_columns(
        df,
        [
            "distance_to_nearest_forest_m",
            "distance_to_nearest_scrub_m",
            "distance_to_nearest_grassland_m",
            "distance_to_nearest_heath_m",
            "distance_to_nearest_natural_vegetation_m"
        ]
    )
)


# ============================================================
# 13. LAND COVER FLAGS
# ============================================================

landcover = (
    df["dw_final_land_cover"]
    .fillna("")
    .astype(str)
    .str.lower()
)


df["lc_built"] = (
    landcover == "built"
)

df["lc_crops"] = (
    landcover == "crops"
)

df["lc_bare"] = (
    landcover == "bare"
)

df["lc_forest"] = (
    landcover == "trees"
)

df["lc_scrub"] = (
    landcover == "shrub_and_scrub"
)

df["lc_grass"] = (
    landcover == "grass"
)

df["lc_natural"] = (
    df["lc_forest"]
    |
    df["lc_scrub"]
    |
    df["lc_grass"]
)


# ============================================================
# 14. PROXIMITY FLAGS
# ============================================================

def near(
    dataframe,
    column,
    threshold
):

    if column not in dataframe.columns:

        return pd.Series(
            False,
            index=dataframe.index
        )


    values = pd.to_numeric(
        dataframe[column],
        errors="coerce"
    )


    return (
        values
        <=
        threshold
    )


# Fossil / gas

df["flare_near_2km"] = near(
    df,
    "distance_to_nearest_flare_m",
    2000
)

df["oil_well_near_2km"] = near(
    df,
    "distance_to_nearest_oil_well_m",
    2000
)

df["industrial_near_1km"] = near(
    df,
    "distance_to_nearest_industrial_m",
    1000
)

df["gasometer_near_2km"] = near(
    df,
    "distance_to_nearest_gasometer_m",
    2000
)


# Mining

df["mine_near_2km"] = near(
    df,
    "distance_to_nearest_mineshaft_m",
    2000
)

df["adit_near_2km"] = near(
    df,
    "distance_to_nearest_adit_m",
    2000
)

df["quarry_near_2km"] = near(
    df,
    "distance_to_nearest_quarry_m",
    2000
)


# Agriculture

df["farmland_near_2km"] = near(
    df,
    "distance_to_nearest_farmland_m",
    2000
)

df["agriculture_near_2km"] = near(
    df,
    "distance_to_nearest_agriculture_m",
    2000
)


# Wildfire

df["forest_near_2km"] = near(
    df,
    "distance_to_nearest_forest_m",
    2000
)

df["scrub_near_2km"] = near(
    df,
    "distance_to_nearest_scrub_m",
    2000
)

df["natural_vegetation_near_2km"] = near(
    df,
    "distance_to_nearest_natural_vegetation_m",
    2000
)


# ============================================================
# 15. INITIAL EVIDENCE SCORES
# ============================================================

df["gas_flare_score"] = 0.0

df["industrial_score"] = 0.0

df["agricultural_score"] = 0.0

df["mining_score"] = 0.0

df["wildfire_score"] = 0.0

df["other_score"] = 1.0


# Gas flare

df["gas_flare_score"] += (
    df["flare_near_2km"].astype(int)
    * 8
)

df["gas_flare_score"] += (
    df["gasometer_near_2km"].astype(int)
    * 5
)

df["gas_flare_score"] += (
    df["oil_well_near_2km"].astype(int)
    * 3
)


# Industrial

df["industrial_score"] += (
    df["industrial_near_1km"].astype(int)
    * 6
)

df["industrial_score"] += (
    df["lc_built"].astype(int)
    * 2
)


# Agricultural

df["agricultural_score"] += (
    df["farmland_near_2km"].astype(int)
    * 4
)

df["agricultural_score"] += (
    df["agriculture_near_2km"].astype(int)
    * 4
)

df["agricultural_score"] += (
    df["lc_crops"].astype(int)
    * 3
)


# Mining

df["mining_score"] += (
    df["quarry_near_2km"].astype(int)
    * 6
)

df["mining_score"] += (
    df["mine_near_2km"].astype(int)
    * 5
)

df["mining_score"] += (
    df["adit_near_2km"].astype(int)
    * 3
)

df["mining_score"] += (
    df["lc_bare"].astype(int)
    * 2
)


# Wildfire

df["wildfire_score"] += (
    df["forest_near_2km"].astype(int)
    * 3
)

df["wildfire_score"] += (
    df["scrub_near_2km"].astype(int)
    * 3
)

df["wildfire_score"] += (
    df["natural_vegetation_near_2km"].astype(int)
    * 2
)

df["wildfire_score"] += (
    df["lc_natural"].astype(int)
    * 3
)


# ============================================================
# 16. FIRE BEHAVIOR EVIDENCE
# ============================================================

df["high_persistence"] = (
    df["persistence_score"] >= 0.30
)

df["very_high_persistence"] = (
    df["persistence_score"] >= 0.60
)

df["night_fire"] = (
    df["night_ratio"] >= 0.50
)

df["strong_night_fire"] = (
    df["night_ratio"] >= 0.75
)

df["high_frp"] = (
    df["frp"] >= 10
)

df["very_high_frp"] = (
    df["frp"] >= 20
)

df["high_brightness"] = (
    df["brightness"] >= 330
)

df["very_high_brightness"] = (
    df["brightness"] >= 340
)


# Gas flare behavior

df["gas_flare_score"] += (
    df["night_fire"].astype(int)
    * 2
)

df["gas_flare_score"] += (
    df["high_persistence"].astype(int)
    * 2
)


# Industrial behavior

df["industrial_score"] += (
    df["high_persistence"].astype(int)
    * 3
)

df["industrial_score"] += (
    df["very_high_persistence"].astype(int)
    * 2
)

df["industrial_score"] += (
    df["night_fire"].astype(int)
    * 1
)

df["industrial_score"] += (
    df["high_frp"].astype(int)
    * 2
)


# Agricultural behavior

df["agricultural_score"] += (
    df["high_frp"].astype(int)
)

df["agricultural_score"] += (
    df["high_brightness"].astype(int)
)


# Mining behavior

df["mining_score"] += (
    df["high_persistence"].astype(int)
    * 2
)

df["mining_score"] += (
    df["very_high_persistence"].astype(int)
    * 2
)


# Wildfire behavior

df["wildfire_score"] += (
    df["high_frp"].astype(int)
    * 2
)

df["wildfire_score"] += (
    df["very_high_frp"].astype(int)
    * 2
)

df["wildfire_score"] += (
    df["high_brightness"].astype(int)
)

df["wildfire_score"] += (
    df["very_high_brightness"].astype(int)
)


# ============================================================
# 17. EVIDENCE SPECIFICITY
# ============================================================

evidence_flags = [
    "flare_near_2km",
    "oil_well_near_2km",
    "industrial_near_1km",
    "quarry_near_2km",
    "mine_near_2km",
    "farmland_near_2km",
    "agriculture_near_2km",
    "forest_near_2km",
    "scrub_near_2km",
    "natural_vegetation_near_2km"
]


for flag in evidence_flags:

    frequency = (
        df[flag]
        .astype(float)
        .mean()
    )


    specificity = (
        1.0
        /
        (
            1.0
            +
            frequency * 5.0
        )
    )


    df[
        flag
        +
        "_specificity"
    ] = specificity


# ============================================================
# 18. ADJUSTED EVIDENCE SCORES
# ============================================================

df["gas_flare_score_adj"] = (
    df["gas_flare_score"]
)

df["industrial_score_adj"] = (
    df["industrial_score"]
)

df["agricultural_score_adj"] = (
    df["agricultural_score"]
)

df["mining_score_adj"] = (
    df["mining_score"]
)

df["wildfire_score_adj"] = (
    df["wildfire_score"]
)

df["other_score_adj"] = (
    df["other_score"]
)


# Gas flare

df["gas_flare_score_adj"] += (
    df["flare_near_2km"].astype(int)
    *
    df["flare_near_2km_specificity"]
    *
    5
)

df["gas_flare_score_adj"] += (
    df["oil_well_near_2km"].astype(int)
    *
    df["oil_well_near_2km_specificity"]
    *
    2
)


# Industrial

df["industrial_score_adj"] += (
    df["industrial_near_1km"].astype(int)
    *
    df["industrial_near_1km_specificity"]
    *
    3
)


# Mining

df["mining_score_adj"] += (
    df["quarry_near_2km"].astype(int)
    *
    df["quarry_near_2km_specificity"]
    *
    3
)

df["mining_score_adj"] += (
    df["mine_near_2km"].astype(int)
    *
    df["mine_near_2km_specificity"]
    *
    4
)


# Agriculture

df["agricultural_score_adj"] += (
    df["farmland_near_2km"].astype(int)
    *
    df["farmland_near_2km_specificity"]
    *
    2
)

df["agricultural_score_adj"] += (
    df["agriculture_near_2km"].astype(int)
    *
    df["agriculture_near_2km_specificity"]
    *
    2
)


# Wildfire

df["wildfire_score_adj"] += (
    df["forest_near_2km"].astype(int)
    *
    df["forest_near_2km_specificity"]
    *
    2
)

df["wildfire_score_adj"] += (
    df["scrub_near_2km"].astype(int)
    *
    df["scrub_near_2km_specificity"]
)

df["wildfire_score_adj"] += (
    df["natural_vegetation_near_2km"].astype(int)
    *
    df["natural_vegetation_near_2km_specificity"]
)


# ============================================================
# 19. FINAL EVIDENCE CLASS
# ============================================================

score_columns = {
    "Gas Flare":
        "gas_flare_score_adj",

    "Industrial Fire":
        "industrial_score_adj",

    "Agricultural Fire":
        "agricultural_score_adj",

    "Mining Activity":
        "mining_score_adj",

    "Wildfire":
        "wildfire_score_adj",

    "Other":
        "other_score_adj"
}


score_df = (
    df[
        list(
            score_columns.values()
        )
    ]
    .copy()
)


score_df.columns = (
    list(
        score_columns.keys()
    )
)


df["highest_evidence_class"] = (
    score_df.idxmax(
        axis=1
    )
)


df["highest_evidence_score"] = (
    score_df.max(
        axis=1
    )
)


df["second_highest_evidence_score"] = (
    score_df.apply(
        lambda row:
        row.nlargest(
            2
        ).iloc[-1],
        axis=1
    )
)


df["evidence_margin"] = (
    df["highest_evidence_score"]
    -
    df["second_highest_evidence_score"]
)


# ============================================================
# 20. CLASS MAP
# ============================================================

CLASS_TO_NUMBER = {
    "Other": 0,
    "Industrial Fire": 1,
    "Gas Flare": 2,
    "Agricultural Fire": 3,
    "Mining Activity": 4,
    "Wildfire": 5
}


NUMBER_TO_CLASS = {
    0: "Other",
    1: "Industrial Fire",
    2: "Gas Flare",
    3: "Agricultural Fire",
    4: "Mining Activity",
    5: "Wildfire"
}


df["fire_class_name"] = (
    df["highest_evidence_class"]
)


df["fire_class"] = (
    df["fire_class_name"]
    .map(
        CLASS_TO_NUMBER
    )
    .astype(int)
)


# ============================================================
# 21. EVIDENCE CONFIDENCE
# ============================================================

def evidence_confidence(
    margin
):

    if margin >= 5:

        return "Very High"

    elif margin >= 3:

        return "High"

    elif margin >= 2:

        return "Medium"

    elif margin > 0:

        return "Low"

    return "Tie"


df["evidence_confidence"] = (
    df["evidence_margin"]
    .apply(
        evidence_confidence
    )
)


# ============================================================
# 22. TRAINING QUALITY
# ============================================================

df["training_quality"] = "Low"


df.loc[
    df["evidence_margin"] >= 5,
    "training_quality"
] = "High"


df.loc[
    (
        df["evidence_margin"] >= 2
    )
    &
    (
        df["evidence_margin"] < 5
    ),
    "training_quality"
] = "Medium"


# ============================================================
# 23. LABEL STATUS
# ============================================================

df["label_status"] = "Uncertain"


df.loc[
    df["training_quality"] == "Medium",
    "label_status"
] = "Candidate"


df.loc[
    df["training_quality"] == "High",
    "label_status"
] = "Trusted"


# ============================================================
# 24. LABEL SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("FINAL PSEUDO LABEL DISTRIBUTION")
print("=" * 70)

print(
    df[
        "fire_class_name"
    ]
    .value_counts()
    .sort_index()
)


print("\nTraining quality:")

print(
    df[
        "training_quality"
    ].value_counts()
)


print("\nLabel status:")

print(
    df[
        "label_status"
    ].value_counts()
)


# ============================================================
# 25. QUALITY DATASET
# ============================================================

quality_mask = (
    df["label_status"]
    .isin(
        [
            "Trusted",
            "Candidate"
        ]
    )
)


quality_df = (
    df.loc[
        quality_mask
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


print("\n" + "=" * 70)
print("QUALITY DATASET")
print("=" * 70)

print(
    "Quality samples:",
    len(quality_df)
)


print("\nClass distribution:")

print(
    quality_df[
        "fire_class_name"
    ]
    .value_counts()
    .sort_index()
)


# ============================================================
# 26. ML FEATURE LIST
# ============================================================

ml_features = [
    "latitude",
    "longitude",
    "brightness",
    "scan",
    "track",
    "bright_t31",
    "frp",

    # OSM distances
    "distance_to_nearest_flare_m",
    "distance_to_nearest_oil_well_m",
    "distance_to_nearest_mineshaft_m",
    "distance_to_nearest_adit_m",
    "distance_to_nearest_gasometer_m",
    "distance_to_nearest_industrial_m",
    "distance_to_nearest_quarry_m",
    "distance_to_nearest_farmland_m",
    "distance_to_nearest_farmyard_m",
    "distance_to_nearest_orchard_m",
    "distance_to_nearest_vineyard_m",
    "distance_to_nearest_plant_nursery_m",
    "distance_to_nearest_greenhouse_m",
    "distance_to_nearest_allotment_m",
    "distance_to_nearest_forest_m",
    "distance_to_nearest_scrub_m",
    "distance_to_nearest_grassland_m",
    "distance_to_nearest_heath_m",
    "distance_to_nearest_agriculture_m",
    "distance_to_nearest_natural_vegetation_m",

    # OSM logs
    "distance_to_nearest_flare_log",
    "distance_to_nearest_oil_well_log",
    "distance_to_nearest_mineshaft_log",
    "distance_to_nearest_adit_log",
    "distance_to_nearest_gasometer_log",
    "distance_to_nearest_industrial_log",
    "distance_to_nearest_quarry_log",
    "distance_to_nearest_farmland_log",
    "distance_to_nearest_farmyard_log",
    "distance_to_nearest_orchard_log",
    "distance_to_nearest_vineyard_log",
    "distance_to_nearest_plant_nursery_log",
    "distance_to_nearest_greenhouse_log",
    "distance_to_nearest_allotment_log",
    "distance_to_nearest_forest_log",
    "distance_to_nearest_scrub_log",
    "distance_to_nearest_grassland_log",
    "distance_to_nearest_heath_log",
    "distance_to_nearest_agriculture_log",
    "distance_to_nearest_natural_vegetation_log",

    # Dynamic World
    "water",
    "trees",
    "grass",
    "flooded_vegetation",
    "crops",
    "shrub_and_scrub",
    "built",
    "bare",
    "snow_and_ice",

    # DW
    "dw_date_difference_days",
    "month",
    "dw_max_probability",
    "dw_available",

    # Persistence
    "detections_7_days",
    "detections_30_days",
    "night_ratio",
    "persistence_score",
    "frp_log",

    # FIRMS confidence
    "confidence_low",
    "confidence_nominal",
    "confidence_high",

    # Time
    "hour_sin",
    "hour_cos",

    # Minimum distances
    "min_fossil_distance_m",
    "min_mining_distance_m",
    "min_agriculture_distance_m",
    "min_wildland_distance_m"
]


# ============================================================
# 27. CHECK FEATURES
# ============================================================

missing_features = [
    c
    for c in ml_features
    if c not in quality_df.columns
]


if missing_features:

    print(
        "Missing features:"
    )

    print(
        missing_features
    )

    raise ValueError(
        "Required ML features are missing."
    )


print("\n" + "=" * 70)
print("FEATURE CHECK")
print("=" * 70)

print(
    "Features:",
    len(ml_features)
)

print(
    "✅ ALL FEATURES AVAILABLE"
)


# ============================================================
# 28. CREATE X / y
# ============================================================

X = (
    quality_df[
        ml_features
    ]
    .copy()
)

y = (
    quality_df[
        "fire_class"
    ]
    .copy()
)


# ============================================================
# 29. NUMERIC CLEANING
# ============================================================

for col in ml_features:

    X[col] = pd.to_numeric(
        X[col],
        errors="coerce"
    )


X = X.replace(
    [
        np.inf,
        -np.inf
    ],
    np.nan
)


# ============================================================
# 30. TRAIN / TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = (
    train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=42,
        stratify=y
    )
)


print("\n" + "=" * 70)
print("TRAIN / TEST")
print("=" * 70)

print(
    "X_train:",
    X_train.shape
)

print(
    "X_test :",
    X_test.shape
)


# ============================================================
# 31. TRAIN-ONLY MEDIAN IMPUTATION
# ============================================================

train_medians = (
    X_train
    .median()
)


X_train = (
    X_train
    .fillna(
        train_medians
    )
    .fillna(0)
)


X_test = (
    X_test
    .fillna(
        train_medians
    )
    .fillna(0)
)


print(
    "Train NaN:",
    X_train.isna().sum().sum()
)

print(
    "Test NaN:",
    X_test.isna().sum().sum()
)


# ============================================================
# 32. CLASS WEIGHTS
# ============================================================

classes_present = (
    np.sort(
        y_train.unique()
    )
)


class_weights_array = (
    compute_class_weight(
        class_weight="balanced",
        classes=classes_present,
        y=y_train
    )
)


class_weights = dict(
    zip(
        classes_present,
        class_weights_array
    )
)


print("\n" + "=" * 70)
print("CLASS WEIGHTS")
print("=" * 70)


for cls, weight in class_weights.items():

    print(
        f"{cls} - "
        f"{NUMBER_TO_CLASS[int(cls)]:20s} "
        f": {weight:.4f}"
    )


# ============================================================
# 33. TRAIN RANDOM FOREST
# ============================================================

print("\n" + "=" * 70)
print("TRAINING RANDOM FOREST")
print("=" * 70)


final_model = RandomForestClassifier(

    n_estimators=200,

    max_depth=None,

    min_samples_leaf=2,

    class_weight=class_weights,

    random_state=42,

    n_jobs=-1
)


final_model.fit(
    X_train,
    y_train
)


print(
    "✅ TRAINING COMPLETE"
)


# ============================================================
# 34. PREDICTION
# ============================================================

y_pred = (
    final_model
    .predict(X_test)
)


# ============================================================
# 35. ACCURACY
# ============================================================

accuracy = (
    accuracy_score(
        y_test,
        y_pred
    )
)


print("\n" + "=" * 70)
print("MODEL ACCURACY")
print("=" * 70)

print(
    f"Accuracy     : "
    f"{accuracy:.4f}"
)

print(
    f"Accuracy (%) : "
    f"{accuracy * 100:.2f}%"
)


# ============================================================
# 36. FIXED CLASSIFICATION REPORT
# ============================================================

labels_present = sorted(
    np.unique(
        np.concatenate(
            [
                np.asarray(
                    y_test
                ),

                np.asarray(
                    y_pred
                )
            ]
        )
    )
)


target_names_present = [
    NUMBER_TO_CLASS[
        int(label)
    ]
    for label in labels_present
]


print("\n" + "=" * 70)
print("CLASSIFICATION REPORT")
print("=" * 70)


print(
    classification_report(
        y_test,
        y_pred,

        labels=labels_present,

        target_names=target_names_present,

        digits=4,

        zero_division=0
    )
)


# ============================================================
# 37. CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(
    y_test,
    y_pred,
    labels=labels_present
)


cm_df = pd.DataFrame(
    cm,

    index=[
        NUMBER_TO_CLASS[
            int(x)
        ]
        for x in labels_present
    ],

    columns=[
        NUMBER_TO_CLASS[
            int(x)
        ]
        for x in labels_present
    ]
)


print("\n" + "=" * 70)
print("CONFUSION MATRIX")
print("=" * 70)

print(cm_df)

print(
    "\nCorrect:",
    np.trace(cm)
)

print(
    "Incorrect:",
    cm.sum()
    -
    np.trace(cm)
)


# ============================================================
# 38. CLASS-WISE METRICS
# ============================================================

precision, recall, f1, support = (
    precision_recall_fscore_support(

        y_test,

        y_pred,

        labels=labels_present,

        zero_division=0
    )
)


metrics_df = pd.DataFrame({

    "Class": [
        NUMBER_TO_CLASS[
            int(x)
        ]
        for x in labels_present
    ],

    "Precision":
        precision,

    "Recall":
        recall,

    "F1":
        f1,

    "Support":
        support
})


print("\n" + "=" * 70)
print("CLASS-WISE METRICS")
print("=" * 70)

print(
    metrics_df.round(4)
)


# ============================================================
# 39. FEATURE IMPORTANCE
# ============================================================

feature_importance = pd.DataFrame({

    "feature":
        ml_features,

    "importance":
        final_model.feature_importances_
})


feature_importance = (
    feature_importance
    .sort_values(
        "importance",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


print("\n" + "=" * 70)
print("TOP 30 FEATURES")
print("=" * 70)


print(
    feature_importance
    .head(30)
    .to_string(
        index=False
    )
)


# ============================================================
# 40. FEATURE GROUP IMPORTANCE
# ============================================================

distance_features = [
    f
    for f in ml_features
    if (
        f.startswith(
            "distance_to_nearest_"
        )
        or
        f.startswith("min_")
    )
]


landcover_features = [
    "water",
    "trees",
    "grass",
    "flooded_vegetation",
    "crops",
    "shrub_and_scrub",
    "built",
    "bare",
    "snow_and_ice"
]


persistence_features = [
    "persistence_score"
]


satellite_features = [
    f
    for f in ml_features
    if (
        f not in distance_features
        and
        f not in landcover_features
        and
        f not in persistence_features
    )
]


def group_importance(
    feature_list
):

    return (
        feature_importance[
            feature_importance[
                "feature"
            ].isin(
                feature_list
            )
        ]["importance"]
        .sum()
    )


group_df = pd.DataFrame({

    "Feature Group": [

        "Satellite / FIRMS",

        "OSM / Distance",

        "Land Cover",

        "Persistence"
    ],

    "Importance": [

        group_importance(
            satellite_features
        ),

        group_importance(
            distance_features
        ),

        group_importance(
            landcover_features
        ),

        group_importance(
            persistence_features
        )
    ]
})


group_df["Percentage"] = (
    group_df["Importance"]
    *
    100
)


print("\n" + "=" * 70)
print("FEATURE GROUP IMPORTANCE")
print("=" * 70)


print(
    group_df
    .sort_values(
        "Importance",
        ascending=False
    )
    .round(4)
    .to_string(
        index=False
    )
)


# ============================================================
# 41. CREATE OUTPUT DIRECTORY
# ============================================================

OUTPUT_DIR = (
    "/kaggle/working/"
    "final_fire_model"
)


os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


# ============================================================
# 42. SAVE MODEL PACKAGE
# ============================================================

model_package = {

    "model":
        final_model,

    "features":
        ml_features,

    "class_mapping":
        NUMBER_TO_CLASS,

    "train_medians":
        train_medians.to_dict(),

    "classes":
        final_model.classes_.tolist(),

    "model_type":
        "Random Forest",

    "random_state":
        42
}


MODEL_PATH = os.path.join(
    OUTPUT_DIR,
    "model_package.joblib"
)


joblib.dump(
    model_package,
    MODEL_PATH
)


# ============================================================
# 43. SAVE MODEL ONLY
# ============================================================

MODEL_ONLY_PATH = os.path.join(
    OUTPUT_DIR,
    "fire_classification_model.joblib"
)


joblib.dump(
    final_model,
    MODEL_ONLY_PATH
)


# ============================================================
# 44. SAVE INFERENCE DATASET
# ============================================================
#
# This is the important file for the separate fast tester.
# ============================================================

tester_columns = [

    # Coordinates
    "latitude",
    "longitude",

    # FIRMS
    "acq_date",
    "acq_time",
    "satellite",
    "instrument",
    "confidence",
    "daynight",
    "scan",
    "track",
    "frp",
    "brightness",
    "bright_t31",

    # Dynamic World
    "dw_final_label",
    "dw_final_land_cover",
    "dw_max_probability",
    "dw_available",

    # Persistence
    "detections_7_days",
    "detections_30_days",
    "night_ratio",
    "persistence_score",

    # Pseudo labels
    "fire_class",
    "fire_class_name",
    "training_quality",
    "label_status",

    # Evidence
    "gas_flare_score_adj",
    "industrial_score_adj",
    "agricultural_score_adj",
    "mining_score_adj",
    "wildfire_score_adj",
    "other_score_adj",

    "highest_evidence_class",
    "highest_evidence_score",
    "second_highest_evidence_score",
    "evidence_margin",
    "evidence_confidence",

    # Minimum distances
    "min_fossil_distance_m",
    "min_mining_distance_m",
    "min_agriculture_distance_m",
    "min_wildland_distance_m",

    # Mining
    "distance_to_nearest_quarry_m",
    "distance_to_nearest_mineshaft_m",
    "distance_to_nearest_adit_m",
    "distance_to_nearest_industrial_m",

    # Agriculture
    "distance_to_nearest_farmland_m",
    "distance_to_nearest_agriculture_m",
    "distance_to_nearest_farmyard_m",
    "distance_to_nearest_orchard_m",

    # Fossil / gas
    "distance_to_nearest_flare_m",
    "distance_to_nearest_oil_well_m",
    "distance_to_nearest_gasometer_m",

    # Wildland
    "distance_to_nearest_forest_m",
    "distance_to_nearest_scrub_m",
    "distance_to_nearest_grassland_m",
    "distance_to_nearest_natural_vegetation_m"
]


tester_columns = [
    c
    for c in tester_columns
    if c in df.columns
]


inference_df = df[
    tester_columns
].copy()


INFERENCE_PATH = os.path.join(
    OUTPUT_DIR,
    "inference_dataset.csv"
)


inference_df.to_csv(
    INFERENCE_PATH,
    index=False
)


# ============================================================
# 45. SAVE METADATA
# ============================================================

metadata = {

    "model_type":
        "Random Forest",

    "n_features":
        len(ml_features),

    "features":
        ml_features,

    "classes":
        final_model.classes_.tolist(),

    "class_mapping":
        NUMBER_TO_CLASS,

    "training_samples":
        len(X_train),

    "testing_samples":
        len(X_test),

    "quality_dataset_size":
        len(quality_df),

    "accuracy":
        float(
            accuracy
        ),

    "important_note":
        "fire_class values are pseudo-labels "
        "generated from contextual evidence; "
        "they are not NASA FIRMS source labels."
}


METADATA_PATH = os.path.join(
    OUTPUT_DIR,
    "model_metadata.joblib"
)


joblib.dump(
    metadata,
    METADATA_PATH
)


# ============================================================
# 46. RELOAD TEST
# ============================================================

print("\n" + "=" * 70)
print("MODEL RELOAD TEST")
print("=" * 70)


loaded_package = joblib.load(
    MODEL_PATH
)


loaded_model = (
    loaded_package[
        "model"
    ]
)


loaded_features = (
    loaded_package[
        "features"
    ]
)


assert (
    loaded_features
    ==
    ml_features
)


original_pred = (
    final_model
    .predict(
        X_test
    )
)


reloaded_pred = (
    loaded_model
    .predict(
        X_test
    )
)


same_predictions = (
    np.array_equal(
        original_pred,
        reloaded_pred
    )
)


print(
    "Predictions identical:",
    same_predictions
)


if not same_predictions:

    raise RuntimeError(
        "Saved model consistency test failed."
    )


print(
    "✅ MODEL CONSISTENCY PASSED"
)


# ============================================================
# 47. FINAL OUTPUT
# ============================================================

print("\n" + "=" * 70)
print("TRAINING COMPLETE")
print("=" * 70)


print(
    "Model package:"
)

print(
    MODEL_PATH
)


print(
    "\nModel only:"
)

print(
    MODEL_ONLY_PATH
)


print(
    "\nInference dataset:"
)

print(
    INFERENCE_PATH
)


print(
    "\nMetadata:"
)

print(
    METADATA_PATH
)


print(
    "\nModel classes:"
)

print(
    final_model.classes_
)


print(
    "\nAccuracy:"
)

print(
    f"{accuracy * 100:.2f}%"
)


print("\nIMPORTANT:")
print(
    "Save/download the entire"
)
print(
    OUTPUT_DIR
)
print(
    "as a Kaggle Dataset before using Notebook 2."
)

SEARCHING FOR INPUT CSV

CSV files found:
/kaggle/input/datasets/rahulkumar53/firms-osm-dynamicworld-full-2025-xyz/firms_osm_dynamicworld_full_2025.csv

Using:
/kaggle/input/datasets/rahulkumar53/firms-osm-dynamicworld-full-2025-xyz/firms_osm_dynamicworld_full_2025.csv

MASTER DATASET
Rows   : 655204
Columns: 77

Date range:
2025-01-01 00:00:00 -> 2025-12-31 00:00:00
Missing datetime: 0

DYNAMIC WORLD
Available: 545858
Unavailable: 109346
Coverage: 83.31 %

CALCULATING PERSISTENCE


Calculating persistence:   0%|          | 0/132 [00:00<?, ?it/s]


Persistence statistics:
       detections_7_days  detections_30_days    night_ratio  persistence_score
count      655204.000000       655204.000000  655204.000000      655204.000000
mean            2.933538            9.806149       0.207770           0.108714
std             8.562552           32.687136       0.378294           0.249617
min             0.000000            0.000000       0.000000           0.000000
25%             0.000000            0.000000       0.000000           0.000000
50%             0.000000            0.000000       0.000000           0.000000
75%             2.000000            2.000000       0.000000           0.078431
max           136.000000          499.000000       1.000000           1.000000

CREATING OSM LOG FEATURES
OSM distance columns: 20

FINAL PSEUDO LABEL DISTRIBUTION
fire_class_name
Agricultural Fire    149372
Gas Flare              6057
Industrial Fire       68109
Mining Activity       62428
Other                  6369
Wildfire             36

In [58]:
# ============================================================
# CELL 2A
# LIVE NASA FIRMS + OSM + BASIC FEATURE ENGINEERING
# ============================================================
#
# RUN AFTER CELL 1
#
# FLOW:
#
# CELL 1
#    ↓
# CELL 2A  ← THIS CELL
#    ↓
# CELL 2A-P
#    ↓
# CELL 2B
#
# THIS CELL:
#   1. Gets user coordinate
#   2. Finds LIVE FIRMS hotspot using WFS
#   3. Uses WFS only to locate the live hotspot
#   4. Uses FIRMS Area API to obtain COMPLETE raw attributes
#   5. Calculates FIRMS engineered features
#   6. Calculates OSM distances
#   7. Creates feature_dict
#
# DOES NOT:
#   - use nearest 2025 FIRMS record
#   - use Earth Engine
#   - calculate persistence
#   - calculate Dynamic World
#
# Cell 2A-P:
#   - calculates 7/30 day persistence
#
# Cell 2B:
#   - adds Dynamic World
#   - creates exact 74-feature vector
#   - predicts using the 2025-trained RF
#
# ============================================================


# ============================================================
# 1. IMPORTS
# ============================================================

import os
import time
import math
import requests
import warnings

import numpy as np
import pandas as pd

from io import StringIO
from datetime import datetime, timezone
from getpass import getpass

warnings.filterwarnings("ignore")


# ============================================================
# 2. VERIFY CELL 1
# ============================================================

required_training_objects = [
    "final_model",
    "ml_features",
    "train_medians",
    "p95_7",
    "p95_30",
    "NUMBER_TO_CLASS"
]

missing_training_objects = [
    x
    for x in required_training_objects
    if x not in globals()
]

if missing_training_objects:

    raise RuntimeError(
        "Cell 1 variables are missing:\n"
        f"{missing_training_objects}\n\n"
        "Run Cell 1 first."
    )

print("✓ Cell 1 variables found")
print(
    f"✓ Model feature count: {len(ml_features)}"
)


# ============================================================
# 3. ENTER USER LOCATION
# ============================================================

print("\n" + "=" * 70)
print("ENTER LOCATION")
print("=" * 70)

LAT = float(
    input("Enter latitude : ").strip()
)

LON = float(
    input("Enter longitude: ").strip()
)


if not (-90 <= LAT <= 90):

    raise ValueError(
        "Latitude must be between -90 and 90."
    )


if not (-180 <= LON <= 180):

    raise ValueError(
        "Longitude must be between -180 and 180."
    )


print(
    "\nQuery location"
)

print(
    f"Latitude : {LAT:.6f}"
)

print(
    f"Longitude: {LON:.6f}"
)


# ============================================================
# 4. NASA FIRMS MAP KEY
# ============================================================

MAP_KEY = os.environ.get(
    "NASA_FIRMS_MAP_KEY"
)


if not MAP_KEY:

    try:

        MAP_KEY = getpass(
            "\nEnter NASA FIRMS MAP_KEY "
            "(input hidden): "
        ).strip()

    except Exception:

        MAP_KEY = input(
            "Enter NASA FIRMS MAP_KEY: "
        ).strip()


if not MAP_KEY:

    raise RuntimeError(
        "NASA FIRMS MAP_KEY is required."
    )


print(
    "✓ NASA FIRMS MAP_KEY received"
)


# ============================================================
# 5. SETTINGS
# ============================================================

# Maximum distance from entered coordinate
# at which a live FIRMS hotspot can be accepted.
LIVE_SEARCH_RADIUS_KM = 10.0

# OSM radius
OSM_RADIUS_KM = 25.0

# Persistence radius used later by Cell 2A-P.
PERSISTENCE_RADIUS_KM = 1.0

# Same missing-distance convention as training.
NOT_FOUND_DISTANCE = 1e9

EARTH_RADIUS_KM = 6371.0088


# ============================================================
# 6. HAVERSINE
# ============================================================

def haversine_vectorized(
    target_lat,
    target_lon,
    lats,
    lons
):
    """
    Distance in kilometers from one coordinate
    to an array of coordinates.
    """

    target_lat_rad = np.radians(
        target_lat
    )

    target_lon_rad = np.radians(
        target_lon
    )

    lat_rad = np.radians(
        np.asarray(
            lats,
            dtype=float
        )
    )

    lon_rad = np.radians(
        np.asarray(
            lons,
            dtype=float
        )
    )

    dlat = (
        lat_rad
        - target_lat_rad
    )

    dlon = (
        lon_rad
        - target_lon_rad
    )

    a = (
        np.sin(
            dlat / 2.0
        ) ** 2

        +

        np.cos(
            target_lat_rad
        )
        *
        np.cos(
            lat_rad
        )
        *
        np.sin(
            dlon / 2.0
        ) ** 2
    )

    return (
        2.0
        *
        EARTH_RADIUS_KM
        *
        np.arcsin(
            np.sqrt(
                np.clip(
                    a,
                    0,
                    1
                )
            )
        )
    )


# ============================================================
# 7. FIRMS SOURCE LIST
# ============================================================

FIRMS_SOURCES = [

    "VIIRS_SNPP_NRT",

    "VIIRS_NOAA20_NRT",

    "VIIRS_NOAA21_NRT"
]


# ============================================================
# 8. BOUNDING BOX
# ============================================================

def make_bbox(
    lat,
    lon,
    radius_km
):

    lat_delta = (
        radius_km / 111.0
    )

    cos_lat = max(
        abs(
            math.cos(
                math.radians(
                    lat
                )
            )
        ),
        0.01
    )

    lon_delta = (
        radius_km
        /
        (
            111.0
            *
            cos_lat
        )
    )

    south = max(
        -90.0,
        lat - lat_delta
    )

    north = min(
        90.0,
        lat + lat_delta
    )

    west = max(
        -180.0,
        lon - lon_delta
    )

    east = min(
        180.0,
        lon + lon_delta
    )

    return (
        west,
        south,
        east,
        north
    )


# ============================================================
# 9. NORMALIZE FIRMS COLUMNS
# ============================================================

def normalize_firms_columns(
    dataframe
):

    if (
        dataframe is None
        or dataframe.empty
    ):

        return pd.DataFrame()


    dataframe = dataframe.copy()


    dataframe.columns = [
        str(c).strip().lower()
        for c in dataframe.columns
    ]


    rename_map = {}


    # FIRMS raw field → model/training field
    if "bright_ti4" in dataframe.columns:

        rename_map[
            "bright_ti4"
        ] = "brightness"


    if "bright_ti5" in dataframe.columns:

        rename_map[
            "bright_ti5"
        ] = "bright_t31"


    dataframe = dataframe.rename(
        columns=rename_map
    )


    return dataframe


# ============================================================
# 10. WFS LAYER MAP
# ============================================================

WFS_LAYER_MAP = {

    "VIIRS_SNPP_NRT":
        "ms:fires_snpp_7days",

    "VIIRS_NOAA20_NRT":
        "ms:fires_noaa20_7days",

    "VIIRS_NOAA21_NRT":
        "ms:fires_noaa21_7days"
}


# ============================================================
# 11. FETCH LIVE FIRMS THROUGH WFS
# ============================================================

def fetch_firms_wfs(
    map_key,
    typename,
    center_lat,
    center_lon,
    radius_km=10.0,
    timeout=90
):

    west, south, east, north = (
        make_bbox(
            center_lat,
            center_lon,
            radius_km
        )
    )


    url = (
        "https://firms.modaps.eosdis.nasa.gov/"
        "mapserver/wfs/South_Asia/"
        f"{map_key}/"
    )


    params = {

        "SERVICE": "WFS",

        "VERSION": "2.0.0",

        "REQUEST": "GetFeature",

        "TYPENAME": typename,

        "OUTPUTFORMAT": "csv",

        "SRSNAME":
            "urn:ogc:def:crs:EPSG::4326",

        "BBOX":
            f"{south},"
            f"{west},"
            f"{north},"
            f"{east}",

        "COUNT": 1000,

        "STARTINDEX": 0
    }


    response = requests.get(
        url,
        params=params,
        timeout=timeout
    )


    response.raise_for_status()


    text = (
        response.text
        .strip()
    )


    if not text:

        return pd.DataFrame()


    if text.lower().startswith(
        "<html"
    ):

        raise RuntimeError(
            "FIRMS WFS returned HTML."
        )


    return normalize_firms_columns(
        pd.read_csv(
            StringIO(text)
        )
    )


# ============================================================
# 12. SEARCH WFS FOR LIVE HOTSPOT
# ============================================================

print("\n" + "=" * 70)
print("SEARCHING LIVE NASA FIRMS — WFS")
print("=" * 70)


wfs_candidates = []


for source in FIRMS_SOURCES:

    typename = (
        WFS_LAYER_MAP[
            source
        ]
    )


    print(
        f"\nRequesting {source}"
    )

    print(
        f"Layer: {typename}"
    )


    try:

        wfs_df = fetch_firms_wfs(

            map_key=MAP_KEY,

            typename=typename,

            center_lat=LAT,

            center_lon=LON,

            radius_km=
                LIVE_SEARCH_RADIUS_KM

        )


        if wfs_df.empty:

            print(
                "  - No detections"
            )

            continue


        # WFS does not necessarily provide
        # satellite/instrument fields.
        # We know them from the layer/source.
        wfs_df[
            "firms_source"
        ] = source


        # Make sure coordinates are numeric.
        wfs_df[
            "latitude"
        ] = pd.to_numeric(
            wfs_df[
                "latitude"
            ],
            errors="coerce"
        )


        wfs_df[
            "longitude"
        ] = pd.to_numeric(
            wfs_df[
                "longitude"
            ],
            errors="coerce"
        )


        # Distance from user coordinate.
        wfs_df[
            "distance_to_query_km"
        ] = haversine_vectorized(

            LAT,

            LON,

            wfs_df[
                "latitude"
            ].values,

            wfs_df[
                "longitude"
            ].values

        )


        wfs_df = wfs_df[
            wfs_df[
                "distance_to_query_km"
            ]
            <= LIVE_SEARCH_RADIUS_KM
        ].copy()


        if wfs_df.empty:

            print(
                "  - No hotspot inside radius"
            )

            continue


        print(
            f"  ✓ {len(wfs_df)} records"
        )


        wfs_candidates.append(
            wfs_df
        )


    except Exception as e:

        print(
            f"  ! WFS failed: {e}"
        )


# ============================================================
# 13. STOP IF WFS FOUND NOTHING
# ============================================================

if not wfs_candidates:

    raise RuntimeError(

        "\nNo LIVE NASA FIRMS hotspot was found "
        f"within {LIVE_SEARCH_RADIUS_KM} km.\n\n"

        f"Latitude : {LAT}\n"
        f"Longitude: {LON}\n\n"

        "If the FIRMS web map shows a hotspot, "
        "retry after a short interval."
    )


# ============================================================
# 14. COMBINE WFS RESULTS
# ============================================================

wfs_all = pd.concat(
    wfs_candidates,
    ignore_index=True
)


# ============================================================
# 15. SELECT CLOSEST WFS HOTSPOT
# ============================================================

wfs_hotspot = (
    wfs_all
    .sort_values(
        "distance_to_query_km"
    )
    .iloc[0]
    .copy()
)


wfs_source = str(
    wfs_hotspot[
        "firms_source"
    ]
)


wfs_lat = float(
    wfs_hotspot[
        "latitude"
    ]
)


wfs_lon = float(
    wfs_hotspot[
        "longitude"
    ]
)


wfs_date = pd.to_datetime(
    wfs_hotspot[
        "acq_date"
    ]
).strftime(
    "%Y-%m-%d"
)


wfs_time = str(
    wfs_hotspot[
        "acq_time"
    ]
)


# ============================================================
# 16. DISPLAY WFS RESULT
# ============================================================

print("\n" + "=" * 70)
print("LIVE FIRMS HOTSPOT LOCATED")
print("=" * 70)

print(
    f"Source          : {wfs_source}"
)

print(
    f"Latitude        : {wfs_lat:.6f}"
)

print(
    f"Longitude       : {wfs_lon:.6f}"
)

print(
    f"Distance        : "
    f"{wfs_hotspot['distance_to_query_km']:.4f} km"
)

print(
    f"Acquisition date: {wfs_date}"
)

print(
    f"Acquisition time: {wfs_time} UTC"
)


# ============================================================
# 17. FETCH COMPLETE RAW FIRMS RECORD
# ============================================================

def fetch_firms_area_exact_date(

    map_key,

    source,

    center_lat,

    center_lon,

    date_str,

    radius_km=2.0,

    timeout=90

):

    """
    Query the FIRMS Area API for the exact source/date
    identified by WFS.

    This is where we obtain the complete raw fields:
        latitude
        longitude
        bright_ti4
        scan
        track
        acq_date
        acq_time
        satellite
        instrument
        confidence
        version
        bright_ti5
        frp
        daynight
    """

    west, south, east, north = (
        make_bbox(

            center_lat,

            center_lon,

            radius_km

        )
    )


    bbox = (

        f"{west:.6f},"

        f"{south:.6f},"

        f"{east:.6f},"

        f"{north:.6f}"

    )


    url = (

        "https://firms.modaps.eosdis.nasa.gov/"

        "api/area/csv/"

        f"{map_key}/"

        f"{source}/"

        f"{bbox}/"

        "1/"

        f"{date_str}"

    )


    response = requests.get(

        url,

        timeout=timeout

    )


    response.raise_for_status()


    text = (
        response.text
        .strip()
    )


    if not text:

        return pd.DataFrame()


    if text.lower().startswith(
        "<html"
    ):

        raise RuntimeError(
            "FIRMS Area API returned HTML."
        )


    return normalize_firms_columns(

        pd.read_csv(
            StringIO(text)
        )

    )


print("\n" + "=" * 70)
print("GETTING COMPLETE FIRMS ATTRIBUTES")
print("=" * 70)

print(
    f"Source: {wfs_source}"
)

print(
    f"Date  : {wfs_date}"
)


try:

    area_df = (
        fetch_firms_area_exact_date(

            map_key=MAP_KEY,

            source=wfs_source,

            center_lat=wfs_lat,

            center_lon=wfs_lon,

            date_str=wfs_date,

            radius_km=2.0

        )
    )

except Exception as e:

    raise RuntimeError(

        "WFS found the live hotspot, but the "
        "FIRMS Area API request failed.\n\n"

        f"Source: {wfs_source}\n"
        f"Date: {wfs_date}\n"

        f"Error: {e}"

    )


if area_df.empty:

    raise RuntimeError(

        "WFS found the live hotspot, but the "
        "FIRMS Area API returned no records "
        "for the same source/date."

    )


print(
    f"✓ Area API returned "
    f"{len(area_df)} records"
)


# ============================================================
# 18. CLEAN AREA DATA
# ============================================================

for col in [

    "latitude",

    "longitude",

    "brightness",

    "scan",

    "track",

    "bright_t31",

    "frp"

]:

    if col in area_df.columns:

        area_df[col] = pd.to_numeric(

            area_df[col],

            errors="coerce"

        )


area_df[
    "acq_date"
] = pd.to_datetime(

    area_df[
        "acq_date"
    ],

    errors="coerce"

)


# ============================================================
# 19. MATCH AREA RECORD TO WFS LOCATION
# ============================================================

area_df[
    "distance_to_wfs_km"
] = haversine_vectorized(

    wfs_lat,

    wfs_lon,

    area_df[
        "latitude"
    ].values,

    area_df[
        "longitude"
    ].values

)


matched_area = area_df[
    area_df[
        "distance_to_wfs_km"
    ]
    <= 1.0
].copy()


# If exact spatial matching produces nothing,
# use the closest Area API record.

if matched_area.empty:

    matched_area = (

        area_df

        .sort_values(
            "distance_to_wfs_km"
        )

        .head(1)

        .copy()

    )


# ============================================================
# 20. SELECT COMPLETE LIVE HOTSPOT
# ============================================================

live_hotspot = (

    matched_area

    .sort_values(
        "distance_to_wfs_km"
    )

    .iloc[0]

    .copy()

)


# ============================================================
# 21. REQUIRED COMPLETE FIRMS FIELDS
# ============================================================

required_live_fields = [

    "latitude",

    "longitude",

    "brightness",

    "scan",

    "track",

    "acq_date",

    "acq_time",

    "satellite",

    "instrument",

    "confidence",

    "version",

    "bright_t31",

    "frp",

    "daynight"

]


missing_live_fields = [

    col

    for col in required_live_fields

    if col not in live_hotspot.index

]


if missing_live_fields:

    raise RuntimeError(

        "The complete FIRMS Area API record "
        "is missing fields:\n"

        f"{missing_live_fields}\n\n"

        "Available fields:\n"

        f"{list(area_df.columns)}"

    )


# ============================================================
# 22. DISPLAY COMPLETE LIVE FIRMS RECORD
# ============================================================

print("\n" + "=" * 70)
print("COMPLETE LIVE NASA FIRMS HOTSPOT")
print("=" * 70)

print(
    f"Latitude       : "
    f"{float(live_hotspot['latitude']):.6f}"
)

print(
    f"Longitude      : "
    f"{float(live_hotspot['longitude']):.6f}"
)

print(
    f"Distance       : "
    f"{float(live_hotspot['distance_to_wfs_km']):.4f} km "
    f"from WFS hotspot"
)

print(
    f"Acquisition date: "
    f"{pd.Timestamp(live_hotspot['acq_date']).strftime('%Y-%m-%d')}"
)

print(
    f"Acquisition time: "
    f"{live_hotspot['acq_time']} UTC"
)

print(
    f"Satellite      : "
    f"{live_hotspot['satellite']}"
)

print(
    f"Instrument     : "
    f"{live_hotspot['instrument']}"
)

print(
    f"Confidence     : "
    f"{live_hotspot['confidence']}"
)

print(
    f"Version        : "
    f"{live_hotspot['version']}"
)

print(
    f"Brightness     : "
    f"{live_hotspot['brightness']}"
)

print(
    f"Bright T31     : "
    f"{live_hotspot['bright_t31']}"
)

print(
    f"FRP            : "
    f"{live_hotspot['frp']}"
)

print(
    f"Day/Night      : "
    f"{live_hotspot['daynight']}"
)


# ============================================================
# 23. PARSE HOTSPOT DATETIME
# ============================================================

acq_date = pd.Timestamp(
    live_hotspot[
        "acq_date"
    ]
)


acq_time_raw = str(
    live_hotspot[
        "acq_time"
    ]
).strip()


if "." in acq_time_raw:

    acq_time_raw = (
        acq_time_raw
        .split(".")[0]
    )


acq_time_raw = (
    acq_time_raw
    .zfill(4)
)


try:

    hour = int(
        acq_time_raw[:2]
    )

    minute = int(
        acq_time_raw[2:4]
    )

except Exception:

    raise RuntimeError(

        "Could not parse FIRMS acquisition time: "

        f"{live_hotspot['acq_time']}"

    )


if (
    hour > 23
    or minute > 59
):

    raise RuntimeError(

        "Invalid FIRMS acquisition time: "

        f"{live_hotspot['acq_time']}"

    )


hotspot_datetime = datetime(

    year=acq_date.year,

    month=acq_date.month,

    day=acq_date.day,

    hour=hour,

    minute=minute,

    tzinfo=timezone.utc

)


print(
    f"\nParsed acquisition UTC: "
    f"{hotspot_datetime.isoformat()}"
)


# ============================================================
# 24. FIRMS ENGINEERED FEATURES
# ============================================================

brightness = float(
    live_hotspot[
        "brightness"
    ]
)


scan = float(
    live_hotspot[
        "scan"
    ]
)


track = float(
    live_hotspot[
        "track"
    ]
)


bright_t31 = float(
    live_hotspot[
        "bright_t31"
    ]
)


frp = float(
    live_hotspot[
        "frp"
    ]
)


# EXACT SAME TRANSFORMATION USED IN TRAINING
frp_log = np.log1p(
    max(
        frp,
        0.0
    )
)


# EXACT SAME CYCLICAL ENCODING USED IN TRAINING
hour_sin = np.sin(
    2
    * np.pi
    * hour
    / 24.0
)


hour_cos = np.cos(
    2
    * np.pi
    * hour
    / 24.0
)


# ============================================================
# 25. CONFIDENCE FEATURES
# ============================================================

confidence_value = str(
    live_hotspot[
        "confidence"
    ]
).strip().lower()


confidence_low = float(
    confidence_value == "l"
)


confidence_nominal = float(
    confidence_value == "n"
)


confidence_high = float(
    confidence_value == "h"
)


print(
    "\nEngineered FIRMS features"
)

print(
    f"FRP log           : "
    f"{frp_log:.6f}"
)

print(
    f"Hour              : "
    f"{hour}"
)

print(
    f"Hour sin          : "
    f"{hour_sin:.6f}"
)

print(
    f"Hour cos          : "
    f"{hour_cos:.6f}"
)

print(
    f"Confidence        : "
    f"{confidence_value}"
)

print(
    f"Confidence low    : "
    f"{confidence_low}"
)

print(
    f"Confidence nominal: "
    f"{confidence_nominal}"
)

print(
    f"Confidence high   : "
    f"{confidence_high}"
)


# ============================================================
# 26. OSM FEATURES
# ============================================================

print("\n" + "=" * 70)
print("CALCULATING OSM DISTANCE FEATURES")
print("=" * 70)


OSM_FEATURE_TAGS = {

    "flare": [
        ("man_made", "flare")
    ],

    "oil_well": [
        ("man_made", "petroleum_well")
    ],

    "mineshaft": [
        ("man_made", "mineshaft")
    ],

    "adit": [
        ("man_made", "adit")
    ],

    "gasometer": [
        ("man_made", "gasometer")
    ],

    "industrial": [
        ("landuse", "industrial")
    ],

    "quarry": [
        ("landuse", "quarry")
    ],

    "farmland": [
        ("landuse", "farmland")
    ],

    "farmyard": [
        ("landuse", "farmyard")
    ],

    "orchard": [
        ("landuse", "orchard")
    ],

    "vineyard": [
        ("landuse", "vineyard")
    ],

    "plant_nursery": [
        ("landuse", "plant_nursery")
    ],

    "greenhouse": [
        ("building", "greenhouse")
    ],

    "allotment": [
        ("landuse", "allotments")
    ],

    "forest": [
        ("landuse", "forest")
    ],

    "scrub": [
        ("natural", "scrub")
    ],

    "grassland": [
        ("natural", "grassland")
    ],

    "heath": [
        ("natural", "heath")
    ],

    "agriculture": [
        ("landuse", "agricultural")
    ],

    "natural_vegetation": [
        ("natural", "vegetation")
    ]
}


# ============================================================
# 27. OSM BBOX
# ============================================================

west, south, east, north = (
    make_bbox(

        float(
            live_hotspot[
                "latitude"
            ]
        ),

        float(
            live_hotspot[
                "longitude"
            ]
        ),

        OSM_RADIUS_KM

    )
)


# ============================================================
# 28. OVERPASS SERVERS
# ============================================================

OVERPASS_URLS = [

    "https://overpass-api.de/api/interpreter",

    "https://overpass.kumi.systems/api/interpreter",

    "https://overpass.private.coffee/api/interpreter"

]


# ============================================================
# 29. BUILD TARGETED OSM QUERY
# ============================================================

def build_overpass_query(
    south,
    west,
    north,
    east
):

    query_parts = []


    osm_tags = [

        ("man_made", "flare"),

        ("man_made", "petroleum_well"),

        ("man_made", "mineshaft"),

        ("man_made", "adit"),

        ("man_made", "gasometer"),

        ("landuse", "industrial"),

        ("landuse", "quarry"),

        ("landuse", "farmland"),

        ("landuse", "farmyard"),

        ("landuse", "orchard"),

        ("landuse", "vineyard"),

        ("landuse", "plant_nursery"),

        ("building", "greenhouse"),

        ("landuse", "allotments"),

        ("landuse", "forest"),

        ("natural", "scrub"),

        ("natural", "grassland"),

        ("natural", "heath"),

        ("landuse", "agricultural"),

        ("natural", "vegetation")

    ]


    for key, value in osm_tags:

        query_parts.append(

            f'node["{key}"="{value}"]'
            f"({south},{west},{north},{east});"

        )


        query_parts.append(

            f'way["{key}"="{value}"]'
            f"({south},{west},{north},{east});"

        )


        query_parts.append(

            f'relation["{key}"="{value}"]'
            f"({south},{west},{north},{east});"

        )


    return f"""
[out:json][timeout:120];
(
{''.join(query_parts)}
);
out center;
"""


overpass_query = (
    build_overpass_query(

        south,

        west,

        north,

        east

    )
)


# ============================================================
# 30. OVERPASS FAILOVER
# ============================================================

def query_overpass_with_failover(

    query,

    servers,

    max_attempts=2,

    wait_seconds=3

):

    headers = {

        "User-Agent":
            "FireSourceClassification-Kaggle/1.0"

    }


    last_error = None


    for server in servers:

        print(
            f"\nTrying Overpass server:"
            f"\n{server}"
        )


        for attempt in range(

            1,

            max_attempts + 1

        ):


            try:

                response = requests.post(

                    server,

                    data=query.encode(
                        "utf-8"
                    ),

                    headers=headers,

                    timeout=180

                )


                # Rate limit
                if response.status_code == 429:

                    print(

                        f"  429 rate limited "
                        f"(attempt {attempt})"

                    )


                    last_error = (

                        f"429 from {server}"

                    )


                    if (
                        attempt
                        < max_attempts
                    ):

                        time.sleep(

                            wait_seconds
                            * attempt

                        )


                    continue


                response.raise_for_status()


                data = response.json()


                print(
                    "  ✓ OSM query successful"
                )


                return data


            except requests.RequestException as e:

                last_error = e


                print(

                    f"  Attempt {attempt} "
                    f"failed: {e}"

                )


                if (
                    attempt
                    < max_attempts
                ):

                    time.sleep(

                        wait_seconds
                        * attempt

                    )


            except Exception as e:

                last_error = e


                print(

                    f"  Unexpected error: "
                    f"{e}"

                )


                if (
                    attempt
                    < max_attempts
                ):

                    time.sleep(

                        wait_seconds
                        * attempt

                    )


    raise RuntimeError(

        "All Overpass servers failed.\n"

        f"Last error: {last_error}"

    )


# ============================================================
# 31. RUN OSM
# ============================================================

osm_data = (
    query_overpass_with_failover(

        overpass_query,

        OVERPASS_URLS

    )
)


# ============================================================
# 32. EXTRACT OSM OBJECTS
# ============================================================

osm_records = []


for element in osm_data.get(

    "elements",

    []

):

    element_type = (
        element.get(
            "type"
        )
    )


    element_id = (
        element.get(
            "id"
        )
    )


    tags = element.get(
        "tags",
        {}
    )


    object_lat = None

    object_lon = None


    if element_type == "node":

        object_lat = (
            element.get(
                "lat"
            )
        )

        object_lon = (
            element.get(
                "lon"
            )
        )


    elif "center" in element:

        object_lat = (
            element[
                "center"
            ].get(
                "lat"
            )
        )

        object_lon = (
            element[
                "center"
            ].get(
                "lon"
            )
        )


    if (

        object_lat is None
        or object_lon is None

    ):

        continue


    osm_records.append({

        "type":
            element_type,

        "id":
            element_id,

        "lat":
            float(
                object_lat
            ),

        "lon":
            float(
                object_lon
            ),

        "tags":
            tags

    })


print(
    f"OSM objects retrieved: "
    f"{len(osm_records)}"
)


# ============================================================
# 33. OSM TAG MATCHING
# ============================================================

def osm_tag_matches(
    tags,
    tag_pairs
):

    for key, value in tag_pairs:

        if (

            str(
                tags.get(
                    key,
                    ""
                )
            ).lower()

            ==

            str(
                value
            ).lower()

        ):

            return True


    return False


# ============================================================
# 34. CALCULATE OSM DISTANCES
# ============================================================

osm_distance_dict = {}


for feature_name, tag_pairs in (
    OSM_FEATURE_TAGS.items()
):

    matching_points = []


    for rec in osm_records:

        if osm_tag_matches(

            rec[
                "tags"
            ],

            tag_pairs

        ):

            matching_points.append(

                (

                    rec[
                        "lat"
                    ],

                    rec[
                        "lon"
                    ]

                )

            )


    if not matching_points:

        distance_m = (
            NOT_FOUND_DISTANCE
        )


    else:

        points = np.asarray(

            matching_points,

            dtype=float

        )


        distances_km = (
            haversine_vectorized(

                float(
                    live_hotspot[
                        "latitude"
                    ]
                ),

                float(
                    live_hotspot[
                        "longitude"
                    ]
                ),

                points[
                    :,
                    0
                ],

                points[
                    :,
                    1
                ]

            )
        )


        distance_m = float(

            distances_km.min()
            * 1000.0

        )


    osm_distance_dict[

        f"distance_to_nearest_"
        f"{feature_name}_m"

    ] = distance_m


# ============================================================
# 35. OSM LOG FEATURES
# ============================================================

for base_feature in list(

    osm_distance_dict.keys()

):

    if base_feature.endswith(
        "_m"
    ):

        log_feature = (

            base_feature[:-2]

            +

            "_log"

        )


        value = float(

            osm_distance_dict[
                base_feature
            ]

        )


        osm_distance_dict[
            log_feature
        ] = np.log1p(
            value
        )


# ============================================================
# 36. MINIMUM DISTANCE FEATURES
# ============================================================

def safe_min(
    feature_names
):

    values = []


    for name in feature_names:

        key = (

            f"distance_to_nearest_"
            f"{name}_m"

        )


        if key not in (
            osm_distance_dict
        ):

            continue


        value = float(

            osm_distance_dict[
                key
            ]

        )


        if (

            np.isfinite(value)

            and

            value
            < NOT_FOUND_DISTANCE

        ):

            values.append(
                value
            )


    if values:

        return float(
            min(values)
        )


    return (
        NOT_FOUND_DISTANCE
    )


# EXACT GROUPING USED IN TRAINING
min_fossil_distance_m = safe_min([

    "flare",

    "oil_well",

    "gasometer"

])


min_mining_distance_m = safe_min([

    "mineshaft",

    "adit",

    "quarry"

])


min_agriculture_distance_m = safe_min([

    "farmland",

    "farmyard",

    "orchard",

    "vineyard",

    "plant_nursery",

    "greenhouse",

    "allotment",

    "agriculture"

])


min_wildland_distance_m = safe_min([

    "forest",

    "scrub",

    "grassland",

    "heath",

    "natural_vegetation"

])


# ============================================================
# 37. DISPLAY OSM RESULTS
# ============================================================

print(
    "\nOSM nearest distances"
)


for feature_name in (
    OSM_FEATURE_TAGS
):

    key = (

        f"distance_to_nearest_"
        f"{feature_name}_m"

    )


    value = (

        osm_distance_dict[
            key
        ]

    )


    if value >= NOT_FOUND_DISTANCE:

        print(

            f"{feature_name:22s}: "
            f"NOT FOUND"

        )

    else:

        print(

            f"{feature_name:22s}: "
            f"{value:.2f} m"

        )


print(
    "\nOSM minimum-distance features"
)


print(

    "min_fossil_distance_m     :",

    min_fossil_distance_m

)


print(

    "min_mining_distance_m     :",

    min_mining_distance_m

)


print(

    "min_agriculture_distance_m:",

    min_agriculture_distance_m

)


print(

    "min_wildland_distance_m   :",

    min_wildland_distance_m

)


# ============================================================
# 38. BUILD FEATURE DICTIONARY
# ============================================================

feature_dict = {}


# ------------------------------------------------------------
# FIRMS FEATURES
# ------------------------------------------------------------

feature_dict[
    "latitude"
] = float(

    live_hotspot[
        "latitude"
    ]

)


feature_dict[
    "longitude"
] = float(

    live_hotspot[
        "longitude"
    ]

)


feature_dict[
    "brightness"
] = brightness


feature_dict[
    "scan"
] = scan


feature_dict[
    "track"
] = track


feature_dict[
    "bright_t31"
] = bright_t31


feature_dict[
    "frp"
] = frp


# ------------------------------------------------------------
# OSM FEATURES
# ------------------------------------------------------------

feature_dict.update(
    osm_distance_dict
)


# ------------------------------------------------------------
# PERSISTENCE PLACEHOLDERS
#
# Cell 2A-P will replace these.
# ------------------------------------------------------------

feature_dict[
    "detections_7_days"
] = 0.0


feature_dict[
    "detections_30_days"
] = 0.0


feature_dict[
    "night_ratio"
] = 0.0


feature_dict[
    "persistence_score"
] = 0.0


# ------------------------------------------------------------
# FIRMS ENGINEERED FEATURES
# ------------------------------------------------------------

feature_dict[
    "frp_log"
] = float(
    frp_log
)


feature_dict[
    "confidence_low"
] = float(
    confidence_low
)


feature_dict[
    "confidence_nominal"
] = float(
    confidence_nominal
)


feature_dict[
    "confidence_high"
] = float(
    confidence_high
)


feature_dict[
    "hour_sin"
] = float(
    hour_sin
)


feature_dict[
    "hour_cos"
] = float(
    hour_cos
)


# ------------------------------------------------------------
# MINIMUM DISTANCE FEATURES
# ------------------------------------------------------------

feature_dict[
    "min_fossil_distance_m"
] = float(
    min_fossil_distance_m
)


feature_dict[
    "min_mining_distance_m"
] = float(
    min_mining_distance_m
)


feature_dict[
    "min_agriculture_distance_m"
] = float(
    min_agriculture_distance_m
)


feature_dict[
    "min_wildland_distance_m"
] = float(
    min_wildland_distance_m
)


# ============================================================
# 39. CREATE CELL 2A DATAFRAME
# ============================================================

cell2a_df = pd.DataFrame(
    [feature_dict]
)


# ============================================================
# 40. DYNAMIC WORLD FIELDS TO BE PROVIDED BY CELL 2B
# ============================================================

dw_features = [

    "water",

    "trees",

    "grass",

    "flooded_vegetation",

    "crops",

    "shrub_and_scrub",

    "built",

    "bare",

    "snow_and_ice",

    "dw_date_difference_days",

    "month",

    "dw_max_probability",

    "dw_available"

]


# ============================================================
# 41. VERIFY NON-DW FEATURES
# ============================================================

missing_model_features = [

    feature

    for feature in ml_features

    if feature not in feature_dict

]


non_dw_missing = [

    feature

    for feature in missing_model_features

    if feature not in dw_features

]


print(
    "\n" + "=" * 70
)

print(
    "CELL 2A FEATURE CHECK"
)

print(
    "=" * 70
)

print(
    f"Current features : "
    f"{len(feature_dict)}"
)

print(
    f"Model features   : "
    f"{len(ml_features)}"
)


if non_dw_missing:

    print(
        "\n❌ Non-DW features missing:"
    )


    for feature in non_dw_missing:

        print(
            f"  - {feature}"
        )


    raise RuntimeError(

        "Cell 2A did not create "
        "all required non-DW features."

    )


print(
    "\nFeatures intentionally "
    "left for Cell 2B:"
)


for feature in dw_features:

    print(
        f"  - {feature}"
    )


print(
    "\n✓ All non-Dynamic-World "
    "features are ready"
)


# ============================================================
# 42. SAVE VARIABLES FOR CELL 2A-P AND CELL 2B
# ============================================================

live_hotspot_lat = float(

    live_hotspot[
        "latitude"
    ]

)


live_hotspot_lon = float(

    live_hotspot[
        "longitude"
    ]

)


live_hotspot_datetime = (
    hotspot_datetime
)


live_firms_row = (
    live_hotspot.copy()
)


# ============================================================
# 43. FINAL OUTPUT
# ============================================================

print(
    "\n" + "=" * 70
)

print(
    "CELL 2A COMPLETED SUCCESSFULLY"
)

print(
    "=" * 70
)

print(
    "✓ LIVE FIRMS hotspot located through WFS"
)

print(
    "✓ Complete FIRMS attributes obtained "
    "through Area API"
)

print(
    "✓ FIRMS bright_ti4 → brightness"
)

print(
    "✓ FIRMS bright_ti5 → bright_t31"
)

print(
    "✓ FRP log calculated"
)

print(
    "✓ Hour sin/cos calculated"
)

print(
    "✓ Confidence features calculated"
)

print(
    "✓ OSM distances calculated"
)

print(
    "✓ OSM log distances calculated"
)

print(
    "✓ Minimum-distance features calculated"
)

print(
    "✓ Persistence placeholders created"
)

print(
    "✓ feature_dict created"
)

print(
    "✓ Ready for Cell 2A-P"
)

print(
    "\nDO NOT run the old "
    "'FINAL SETUP FOR FAST USER-POINT TESTING' "
    "cell."
)

✓ Cell 1 variables found
✓ Model feature count: 74

ENTER LOCATION


Enter latitude :  23.8053
Enter longitude:  85.53831



Query location
Latitude : 23.805300
Longitude: 85.538310



Enter NASA FIRMS MAP_KEY (input hidden):  ········


✓ NASA FIRMS MAP_KEY received

SEARCHING LIVE NASA FIRMS — WFS

Requesting VIIRS_SNPP_NRT
Layer: ms:fires_snpp_7days
  ✓ 9 records

Requesting VIIRS_NOAA20_NRT
Layer: ms:fires_noaa20_7days
  ✓ 14 records

Requesting VIIRS_NOAA21_NRT
Layer: ms:fires_noaa21_7days
  ✓ 7 records

LIVE FIRMS HOTSPOT LOCATED
Source          : VIIRS_NOAA20_NRT
Latitude        : 23.805300
Longitude       : 85.538310
Distance        : 0.0000 km
Acquisition date: 2026-09-08
Acquisition time: 2008 UTC

GETTING COMPLETE FIRMS ATTRIBUTES
Source: VIIRS_NOAA20_NRT
Date  : 2026-09-08
✓ Area API returned 2 records

COMPLETE LIVE NASA FIRMS HOTSPOT
Latitude       : 23.805300
Longitude      : 85.538310
Distance       : 0.0000 km from WFS hotspot
Acquisition date: 2026-09-08
Acquisition time: 2008 UTC
Satellite      : N20
Instrument     : VIIRS
Confidence     : n
Version        : 2.0NRT
Brightness     : 301.78
Bright T31     : 289.46
FRP            : 0.96
Day/Night      : N

Parsed acquisition UTC: 2026-09-08T20:08:00+00:

In [59]:
# ============================================================
# CELL 2A-P — CORRECT FIRMS PERSISTENCE
# ============================================================
#
# Run AFTER Cell 2A has completed successfully.
#
# This cell fixes the previous problem:
#   FIRMS Area API supports only 1-5 days/request.
#
# It fetches the 30-day history in valid chunks and then
# reproduces the original training persistence logic:
#
#   - within 1 km
#   - previous detections only
#   - detections_7_days
#   - detections_30_days
#   - night_ratio
#   - persistence_score
#
# ============================================================

import numpy as np
import pandas as pd
import requests
from io import StringIO
from datetime import timedelta


# ============================================================
# 1. VERIFY CELL 2A VARIABLES
# ============================================================

required_vars = [
    "MAP_KEY",
    "live_hotspot",
    "live_hotspot_datetime",
    "FIRMS_SOURCES",
    "PERSISTENCE_RADIUS_KM",
    "haversine_vectorized",
    "feature_dict",
    "p95_7",
    "p95_30"
]

missing_vars = [
    x for x in required_vars
    if x not in globals()
]

if missing_vars:
    raise RuntimeError(
        "Required Cell 2A variables are missing:\n"
        f"{missing_vars}\n\n"
        "Run Cell 2A first."
    )

print("✓ Cell 2A variables found")
print("✓ Original training p95 values found")


# ============================================================
# 2. TARGET LIVE HOTSPOT
# ============================================================

target_lat = float(
    live_hotspot["latitude"]
)

target_lon = float(
    live_hotspot["longitude"]
)

target_datetime = (
    pd.Timestamp(
        live_hotspot_datetime
    )
)

if target_datetime.tzinfo is None:
    target_datetime = target_datetime.tz_localize("UTC")
else:
    target_datetime = target_datetime.tz_convert("UTC")


target_date = (
    target_datetime
    .tz_convert(None)
    .normalize()
)

print("\n" + "=" * 65)
print("CORRECTING FIRMS PERSISTENCE")
print("=" * 65)

print(
    f"Target hotspot : "
    f"{target_lat:.6f}, {target_lon:.6f}"
)

print(
    f"Target datetime: "
    f"{target_datetime.isoformat()}"
)


# ============================================================
# 3. FIRMS CHUNK FETCHER
# ============================================================

def fetch_firms_history_chunk(
    source,
    start_date,
    day_count
):
    """
    Fetch at most 5 days from NASA FIRMS.

    NASA FIRMS Area API:
        /area/csv/MAP_KEY/SOURCE/BBOX/DAY_RANGE/DATE
    """

    if not (1 <= day_count <= 5):
        raise ValueError(
            "FIRMS DAY_RANGE must be between 1 and 5."
        )

    # Use a small spatial box around the live hotspot.
    radius_km = PERSISTENCE_RADIUS_KM

    lat_delta = (
        radius_km / 111.0
    )

    cos_lat = max(
        abs(
            np.cos(
                np.radians(target_lat)
            )
        ),
        0.01
    )

    lon_delta = (
        radius_km
        / (111.0 * cos_lat)
    )

    south = max(
        -90.0,
        target_lat - lat_delta
    )

    north = min(
        90.0,
        target_lat + lat_delta
    )

    west = max(
        -180.0,
        target_lon - lon_delta
    )

    east = min(
        180.0,
        target_lon + lon_delta
    )

    bbox = (
        f"{west:.6f},"
        f"{south:.6f},"
        f"{east:.6f},"
        f"{north:.6f}"
    )

    url = (
        "https://firms.modaps.eosdis.nasa.gov/"
        "api/area/csv/"
        f"{MAP_KEY}/"
        f"{source}/"
        f"{bbox}/"
        f"{day_count}/"
        f"{start_date.strftime('%Y-%m-%d')}"
    )

    response = requests.get(
        url,
        timeout=90
    )

    response.raise_for_status()

    text = response.text.strip()

    if not text:
        return pd.DataFrame()

    if text.lower().startswith("<html"):
        raise RuntimeError(
            "FIRMS returned HTML instead of CSV."
        )

    df_chunk = pd.read_csv(
        StringIO(text)
    )

    return df_chunk


# ============================================================
# 4. NORMALIZE FIRMS COLUMNS
# ============================================================

def normalize_history_columns(df):

    if df is None or df.empty:
        return pd.DataFrame()

    df = df.copy()

    df.columns = [
        str(c).strip()
        for c in df.columns
    ]

    rename_map = {}

    if "bright_ti4" in df.columns:
        rename_map[
            "bright_ti4"
        ] = "brightness"

    if "bright_ti5" in df.columns:
        rename_map[
            "bright_ti5"
        ] = "bright_t31"

    df = df.rename(
        columns=rename_map
    )

    return df


# ============================================================
# 5. BUILD 30-DAY DATE WINDOWS
# ============================================================
#
# We need dates from:
#
#   target_date - 29 days
#   through
#   target_date
#
# Because the final filtering below uses the exact
# target timestamp and removes the current/future
# detections.
#
# Each request is <= 5 days.

history_start = (
    target_date
    - pd.Timedelta(days=29)
)

history_end = target_date

date_windows = []

window_start = history_start

while window_start <= history_end:

    remaining_days = (
        history_end
        - window_start
    ).days + 1

    day_count = min(
        5,
        remaining_days
    )

    date_windows.append(
        (
            window_start,
            day_count
        )
    )

    window_start = (
        window_start
        + pd.Timedelta(
            days=day_count
        )
    )


print(
    f"\n30-day period: "
    f"{history_start.strftime('%Y-%m-%d')}"
    f" -> "
    f"{history_end.strftime('%Y-%m-%d')}"
)

print(
    f"FIRMS API chunks required: "
    f"{len(date_windows)} per source"
)


# ============================================================
# 6. DOWNLOAD HISTORY
# ============================================================

history_frames = []

for source in FIRMS_SOURCES:

    print(
        f"\nSource: {source}"
    )

    for chunk_number, (
        chunk_start,
        day_count
    ) in enumerate(
        date_windows,
        start=1
    ):

        chunk_end = (
            chunk_start
            + pd.Timedelta(
                days=day_count - 1
            )
        )

        print(
            f"  Chunk {chunk_number:02d}: "
            f"{chunk_start.strftime('%Y-%m-%d')}"
            f" -> "
            f"{chunk_end.strftime('%Y-%m-%d')}"
            f" ({day_count} days)"
        )

        try:

            chunk_df = (
                fetch_firms_history_chunk(
                    source=source,
                    start_date=chunk_start,
                    day_count=day_count
                )
            )

            if chunk_df.empty:

                print(
                    "    - No records"
                )

                continue

            chunk_df[
                "firms_source"
            ] = source

            history_frames.append(
                chunk_df
            )

            print(
                f"    ✓ {len(chunk_df)} records"
            )

        except Exception as e:

            print(
                f"    ! Request failed: {e}"
            )


# ============================================================
# 7. COMBINE HISTORY
# ============================================================

if not history_frames:

    raise RuntimeError(
        "No FIRMS historical records could be "
        "downloaded for persistence calculation."
    )


firms_history = pd.concat(
    history_frames,
    ignore_index=True
)

firms_history = normalize_history_columns(
    firms_history
)

print(
    f"\nCombined history records: "
    f"{len(firms_history)}"
)


# ============================================================
# 8. CLEAN HISTORY
# ============================================================

for col in [
    "latitude",
    "longitude"
]:

    firms_history[col] = pd.to_numeric(
        firms_history[col],
        errors="coerce"
    )


firms_history["acq_date"] = (
    pd.to_datetime(
        firms_history["acq_date"],
        errors="coerce"
    )
)


firms_history["acq_time"] = (
    firms_history["acq_time"]
    .astype(str)
    .str.replace(
        ".0",
        "",
        regex=False
    )
    .str.zfill(4)
)


firms_history = firms_history.dropna(
    subset=[
        "latitude",
        "longitude",
        "acq_date"
    ]
).copy()


# ============================================================
# 9. REMOVE DUPLICATES
# ============================================================

history_dedup_cols = [
    "latitude",
    "longitude",
    "acq_date",
    "acq_time",
    "satellite",
    "instrument"
]

history_dedup_cols = [
    c
    for c in history_dedup_cols
    if c in firms_history.columns
]

firms_history = (
    firms_history
    .drop_duplicates(
        subset=history_dedup_cols
    )
    .reset_index(drop=True)
)

print(
    f"Records after duplicate removal: "
    f"{len(firms_history)}"
)


# ============================================================
# 10. BUILD EXACT FIRMS DATETIME
# ============================================================

def parse_firms_datetime(row):

    date_value = row["acq_date"]

    time_value = str(
        row["acq_time"]
    ).strip()

    if "." in time_value:
        time_value = (
            time_value
            .split(".")[0]
        )

    time_value = time_value.zfill(4)

    try:

        hh = int(
            time_value[:2]
        )

        mm = int(
            time_value[2:4]
        )

        if hh > 23 or mm > 59:
            return pd.NaT

        return pd.Timestamp(
            year=date_value.year,
            month=date_value.month,
            day=date_value.day,
            hour=hh,
            minute=mm,
            tz="UTC"
        )

    except Exception:

        return pd.NaT


firms_history[
    "datetime"
] = firms_history.apply(
    parse_firms_datetime,
    axis=1
)


firms_history = firms_history.dropna(
    subset=["datetime"]
).copy()


# ============================================================
# 11. SPATIAL FILTER — SAME 1 KM AS TRAINING
# ============================================================

firms_history[
    "distance_km"
] = haversine_vectorized(
    target_lat,
    target_lon,
    firms_history["latitude"].values,
    firms_history["longitude"].values
)


nearby_history = firms_history[
    firms_history["distance_km"]
    <= PERSISTENCE_RADIUS_KM
].copy()


print(
    f"\nRecords within "
    f"{PERSISTENCE_RADIUS_KM:.1f} km: "
    f"{len(nearby_history)}"
)


# ============================================================
# 12. STRICTLY PREVIOUS DETECTIONS
# ============================================================
#
# This is IMPORTANT.
#
# Your original training code uses:
#
#       neighbor_times < target_time
#
# therefore the current live FIRMS observation itself
# must NOT count as persistence.
#
# This also removes any later observation on the same
# acquisition date.

previous_history = nearby_history[
    nearby_history["datetime"]
    < target_datetime
].copy()


# ============================================================
# 13. EXACT 30-DAY WINDOW
# ============================================================

window_30_start = (
    target_datetime
    - pd.Timedelta(
        days=30
    )
)

window_7_start = (
    target_datetime
    - pd.Timedelta(
        days=7
    )
)


previous_30 = previous_history[
    previous_history["datetime"]
    >= window_30_start
].copy()


previous_7 = previous_history[
    previous_history["datetime"]
    >= window_7_start
].copy()


# ============================================================
# 14. COUNTS
# ============================================================

detections_7_days = int(
    len(previous_7)
)

detections_30_days = int(
    len(previous_30)
)


# ============================================================
# 15. NIGHT RATIO
# ============================================================

if len(previous_30) > 0:

    night_values = (
        previous_30["daynight"]
        .astype(str)
        .str.upper()
        .str.strip()
    )

    night_ratio = float(
        (
            night_values == "N"
        ).mean()
    )

else:

    night_ratio = 0.0


# ============================================================
# 16. USE ORIGINAL TRAINING P95 VALUES
# ============================================================
#
# DO NOT recompute p95 from this tiny local history.
#
# The Random Forest was trained using p95 values calculated
# from the complete 2025 training dataset.
#
# The original training variables are:
#   p95_7
#   p95_30
#
# ============================================================

training_p95_7 = float(
    p95_7
)

training_p95_30 = float(
    p95_30
)


if training_p95_7 <= 0:
    training_p95_7 = 1.0

if training_p95_30 <= 0:
    training_p95_30 = 1.0


# ============================================================
# 17. PERSISTENCE SCORE
# ============================================================

score_7 = (
    detections_7_days
    / training_p95_7
)

score_30 = (
    detections_30_days
    / training_p95_30
)


score_7 = np.clip(
    score_7,
    0,
    1
)

score_30 = np.clip(
    score_30,
    0,
    1
)


persistence_score = (
    0.6 * score_7
    +
    0.4 * score_30
)


# ============================================================
# 18. DISPLAY CORRECTED VALUES
# ============================================================

print("\n" + "=" * 65)
print("CORRECTED FIRMS PERSISTENCE")
print("=" * 65)

print(
    f"Previous 7-day detections : "
    f"{detections_7_days}"
)

print(
    f"Previous 30-day detections: "
    f"{detections_30_days}"
)

print(
    f"Night ratio               : "
    f"{night_ratio:.6f}"
)

print(
    f"Training P95 7            : "
    f"{training_p95_7:.6f}"
)

print(
    f"Training P95 30           : "
    f"{training_p95_30:.6f}"
)

print(
    f"Normalized 7-day score    : "
    f"{score_7:.6f}"
)

print(
    f"Normalized 30-day score   : "
    f"{score_30:.6f}"
)

print(
    f"Persistence score         : "
    f"{persistence_score:.6f}"
)


# ============================================================
# 19. UPDATE FEATURE DICTIONARY
# ============================================================

feature_dict[
    "detections_7_days"
] = float(
    detections_7_days
)

feature_dict[
    "detections_30_days"
] = float(
    detections_30_days
)

feature_dict[
    "night_ratio"
] = float(
    night_ratio
)

feature_dict[
    "persistence_score"
] = float(
    persistence_score
)


# ============================================================
# 20. UPDATE CELL 2A DATAFRAME
# ============================================================

cell2a_df = pd.DataFrame(
    [feature_dict]
)


# ============================================================
# 21. FINAL CHECK
# ============================================================

print("\n" + "=" * 65)
print("PERSISTENCE UPDATE COMPLETE")
print("=" * 65)

print(
    "✓ 30-day history fetched in valid FIRMS chunks"
)

print(
    "✓ Same 1 km spatial radius as training"
)

print(
    "✓ Only previous detections counted"
)

print(
    "✓ Training p95 values reused"
)

print(
    "✓ persistence_score updated"
)

print(
    "✓ feature_dict updated"
)

print(
    "\nCell 2B can now use the corrected "
    "persistence features."
)

✓ Cell 2A variables found
✓ Original training p95 values found

CORRECTING FIRMS PERSISTENCE
Target hotspot : 23.805300, 85.538310
Target datetime: 2026-09-08T20:08:00+00:00

30-day period: 2026-08-10 -> 2026-09-08
FIRMS API chunks required: 6 per source

Source: VIIRS_SNPP_NRT
  Chunk 01: 2026-08-10 -> 2026-08-14 (5 days)
    - No records
  Chunk 02: 2026-08-15 -> 2026-08-19 (5 days)
    - No records
  Chunk 03: 2026-08-20 -> 2026-08-24 (5 days)
    - No records
  Chunk 04: 2026-08-25 -> 2026-08-29 (5 days)
    - No records
  Chunk 05: 2026-08-30 -> 2026-09-03 (5 days)
    - No records
  Chunk 06: 2026-09-04 -> 2026-09-08 (5 days)
    - No records

Source: VIIRS_NOAA20_NRT
  Chunk 01: 2026-08-10 -> 2026-08-14 (5 days)
    - No records
  Chunk 02: 2026-08-15 -> 2026-08-19 (5 days)
    - No records
  Chunk 03: 2026-08-20 -> 2026-08-24 (5 days)
    - No records
  Chunk 04: 2026-08-25 -> 2026-08-29 (5 days)
    - No records
  Chunk 05: 2026-08-30 -> 2026-09-03 (5 days)
    - No records
  

In [60]:
# ================================================================
# CELL 2B — LIVE DYNAMIC WORLD RETRIEVAL
# TRAINING-COMPATIBLE MISSING-DW HANDLING
# ================================================================

import sys
import subprocess
import importlib.util
import numpy as np
import pandas as pd

print("\n" + "=" * 70)
print("CELL 2B — LIVE DYNAMIC WORLD RETRIEVAL")
print("=" * 70)


# ================================================================
# 1. VERIFY REQUIRED VARIABLES
# ================================================================

required_vars = [
    "feature_dict",
    "live_hotspot_lat",
    "live_hotspot_lon",
    "live_hotspot_datetime",
]

missing = [
    v for v in required_vars
    if v not in globals()
]

if missing:
    raise RuntimeError(
        "Missing required variables: "
        + ", ".join(missing)
        + "\nRun Cell 2A and Cell 2A-P first."
    )

print("✓ Cell 2A / Cell 2A-P variables found")

print(
    f"Hotspot latitude : {live_hotspot_lat:.6f}"
)

print(
    f"Hotspot longitude: {live_hotspot_lon:.6f}"
)

print(
    f"Hotspot datetime : {live_hotspot_datetime}"
)


# ================================================================
# 2. DYNAMIC WORLD FEATURES
# ================================================================

DW_PROBABILITY_BANDS = [
    "water",
    "trees",
    "grass",
    "flooded_vegetation",
    "crops",
    "shrub_and_scrub",
    "built",
    "bare",
    "snow_and_ice",
]

DW_FEATURES = [
    "water",
    "trees",
    "grass",
    "flooded_vegetation",
    "crops",
    "shrub_and_scrub",
    "built",
    "bare",
    "snow_and_ice",
    "dw_date_difference_days",
    "month",
    "dw_max_probability",
    "dw_available",
]


# ================================================================
# 3. EARTH ENGINE API
# ================================================================

print("\n" + "-" * 70)
print("CHECKING EARTH ENGINE PYTHON API")
print("-" * 70)

if importlib.util.find_spec("ee") is None:

    print("Earth Engine API not installed.")
    print("Installing...")

    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "earthengine-api"
    ])

    print("✓ Earth Engine API installed")

else:

    print("✓ Earth Engine API already installed")

import ee


# ================================================================
# 4. INITIALIZE EARTH ENGINE
# ================================================================

EE_PROJECT = "planar-depth-508020-q9"

print("\n" + "-" * 70)
print("INITIALIZING GOOGLE EARTH ENGINE")
print("-" * 70)

print(
    "Earth Engine project:",
    EE_PROJECT
)

try:

    ee.Initialize(
        project=EE_PROJECT
    )

    print(
        "✓ Earth Engine initialized successfully"
    )

    print(
        "✓ Project ID:",
        EE_PROJECT
    )

except Exception as e:

    raise RuntimeError(
        "\nEarth Engine initialization failed.\n\n"
        + str(e)
    )


# ================================================================
# 5. CREATE HOTSPOT POINT
# ================================================================

point = ee.Geometry.Point([
    float(live_hotspot_lon),
    float(live_hotspot_lat)
])

print("\n✓ Hotspot point created")


# ================================================================
# 6. PREPARE FIRMS DATETIME
# ================================================================

target_dt = pd.Timestamp(
    live_hotspot_datetime
)

if target_dt.tzinfo is None:

    target_dt = target_dt.tz_localize(
        "UTC"
    )

else:

    target_dt = target_dt.tz_convert(
        "UTC"
    )

print("\n" + "-" * 70)
print("FIRMS TARGET")
print("-" * 70)

print(
    "FIRMS datetime:",
    target_dt
)

print(
    "FIRMS date    :",
    target_dt.strftime("%Y-%m-%d")
)


# ================================================================
# 7. LOAD DYNAMIC WORLD
# ================================================================

print("\n" + "-" * 70)
print("LOADING DYNAMIC WORLD")
print("-" * 70)

DW = ee.ImageCollection(
    "GOOGLE/DYNAMICWORLD/V1"
)

print(
    "✓ Dynamic World collection loaded"
)


# ================================================================
# 8. SEARCH ±15 DAYS
# ================================================================

SEARCH_DAYS_BEFORE = 15
SEARCH_DAYS_AFTER = 1

window_start = (
    target_dt -
    pd.Timedelta(
        days=SEARCH_DAYS_BEFORE
    )
).strftime("%Y-%m-%d")

window_end = (
    target_dt +
    pd.Timedelta(
        days=SEARCH_DAYS_AFTER
    )
).strftime("%Y-%m-%d")

print("\n" + "-" * 70)
print("SEARCHING DYNAMIC WORLD")
print("-" * 70)

print(
    "Search start:",
    window_start
)

print(
    "Search end  :",
    window_end
)

collection = (
    DW
    .filterBounds(point)
    .filterDate(
        window_start,
        window_end
    )
)

count = collection.size().getInfo()

print(
    "Dynamic World images found:",
    count
)


# ================================================================
# 9. IF NO IMAGES EXIST
# ================================================================

if count == 0:

    print("\n" + "=" * 70)
    print("NO DYNAMIC WORLD IMAGE FOUND")
    print("=" * 70)

    print(
        "Dynamic World is unavailable for this hotspot."
    )

    # ------------------------------------------------------------
    # Training-compatible missing values
    # ------------------------------------------------------------

    for band in DW_PROBABILITY_BANDS:

        feature_dict[band] = np.nan

    feature_dict[
        "dw_date_difference_days"
    ] = np.nan

    feature_dict[
        "month"
    ] = int(target_dt.month)

    feature_dict[
        "dw_max_probability"
    ] = np.nan

    feature_dict[
        "dw_available"
    ] = 0

    print(
        "\n✓ dw_available = 0"
    )

    print(
        "✓ Missing DW numerical fields stored as NaN"
    )

    print(
        "✓ Cell 2C will apply training medians"
    )

else:

    # ============================================================
    # 10. GET IMAGE METADATA
    # ============================================================

    image_list = collection.toList(
        count
    )

    candidates = []

    for i in range(count):

        img = ee.Image(
            image_list.get(i)
        )

        try:

            timestamp = (
                img
                .get(
                    "system:time_start"
                )
                .getInfo()
            )

            if timestamp is None:
                continue

            image_dt = pd.to_datetime(
                timestamp,
                unit="ms",
                utc=True
            )

            difference_days = abs(
                (
                    image_dt -
                    target_dt
                ).total_seconds()
            ) / 86400.0

            image_id = (
                img
                .get("system:index")
                .getInfo()
            )

            candidates.append({
                "image": img,
                "image_dt": image_dt,
                "difference_days":
                    difference_days,
                "image_id":
                    image_id
            })

        except Exception:
            continue

    candidates.sort(
        key=lambda x:
        x["difference_days"]
    )


    # ============================================================
    # 11. FIND FIRST VALID PIXEL
    # ============================================================

    print("\n" + "-" * 70)
    print("CHECKING FOR VALID DYNAMIC WORLD PIXEL")
    print("-" * 70)

    valid_dw = None

    for idx, candidate in enumerate(
        candidates,
        start=1
    ):

        print(
            f"\nCandidate "
            f"{idx}/{len(candidates)}"
        )

        print(
            "Image ID      :",
            candidate["image_id"]
        )

        print(
            "Image datetime:",
            candidate["image_dt"]
        )

        print(
            "Date gap      :",
            f"{candidate['difference_days']:.6f}"
        )

        try:

            sample = (
                candidate["image"]
                .select(
                    DW_PROBABILITY_BANDS
                )
                .reduceRegion(
                    reducer=ee.Reducer.first(),
                    geometry=point,
                    scale=10,
                    bestEffort=True
                )
            )

            values = sample.getInfo()

            missing_bands = [
                band
                for band in
                DW_PROBABILITY_BANDS
                if values.get(band) is None
            ]

            if missing_bands:

                print(
                    "  ✗ Pixel masked"
                )

                continue

            valid_dw = {
                "image":
                    candidate["image"],

                "image_dt":
                    candidate["image_dt"],

                "difference_days":
                    candidate[
                        "difference_days"
                    ],

                "image_id":
                    candidate["image_id"],

                "values":
                    values
            }

            print(
                "  ✓ VALID PIXEL FOUND"
            )

            break

        except Exception as e:

            print(
                "  ✗ Pixel read error:",
                e
            )


    # ============================================================
    # 12. NO VALID PIXEL
    # ============================================================

    if valid_dw is None:

        print("\n" + "=" * 70)
        print("NO VALID DYNAMIC WORLD PIXEL")
        print("=" * 70)

        print(
            f"Checked {len(candidates)} "
            f"candidate image(s)."
        )

        print(
            "The hotspot is masked/unavailable."
        )

        # --------------------------------------------------------
        # IMPORTANT:
        # Match original training preprocessing.
        # --------------------------------------------------------

        for band in DW_PROBABILITY_BANDS:

            feature_dict[band] = np.nan

        feature_dict[
            "dw_date_difference_days"
        ] = np.nan

        feature_dict[
            "month"
        ] = int(target_dt.month)

        feature_dict[
            "dw_max_probability"
        ] = np.nan

        feature_dict[
            "dw_available"
        ] = 0

        print(
            "\n✓ Dynamic World marked unavailable"
        )

        print(
            "✓ dw_available = 0"
        )

        print(
            "✓ DW numerical features = NaN"
        )

        print(
            "✓ Cell 2C will use train_medians"
        )


    # ============================================================
    # 13. VALID PIXEL FOUND
    # ============================================================

    else:

        best_dt = valid_dw[
            "image_dt"
        ]

        best_id = valid_dw[
            "image_id"
        ]

        dw_values = valid_dw[
            "values"
        ]

        print("\n" + "=" * 70)
        print("VALID DYNAMIC WORLD IMAGE SELECTED")
        print("=" * 70)

        print(
            "Image ID:",
            best_id
        )

        print(
            "DW datetime:",
            best_dt
        )

        print(
            "Absolute date gap:",
            f"{valid_dw['difference_days']:.6f}"
        )

        # --------------------------------------------------------
        # Convert probabilities
        # --------------------------------------------------------

        dw_probs = {
            band:
            float(dw_values[band])
            for band in
            DW_PROBABILITY_BANDS
        }

        probability_sum = sum(
            dw_probs.values()
        )

        dw_max_probability = max(
            dw_probs.values()
        )

        print("\nDynamic World probabilities:")

        for band in DW_PROBABILITY_BANDS:

            print(
                f"{band:<22}: "
                f"{dw_probs[band]:.6f}"
            )

        print(
            "\nProbability sum:",
            f"{probability_sum:.6f}"
        )

        print(
            "Maximum probability:",
            f"{dw_max_probability:.6f}"
        )

        # --------------------------------------------------------
        # Training-compatible SIGNED date difference
        # --------------------------------------------------------

        target_date_only = (
            target_dt.normalize()
        )

        dw_date_only = (
            best_dt.normalize()
        )

        dw_date_difference_days = (
            target_date_only -
            dw_date_only
        ).total_seconds() / 86400.0

        # --------------------------------------------------------
        # Update feature_dict
        # --------------------------------------------------------

        for band in DW_PROBABILITY_BANDS:

            feature_dict[band] = (
                dw_probs[band]
            )

        feature_dict[
            "dw_date_difference_days"
        ] = float(
            dw_date_difference_days
        )

        feature_dict[
            "month"
        ] = int(
            target_dt.month
        )

        feature_dict[
            "dw_max_probability"
        ] = float(
            dw_max_probability
        )

        feature_dict[
            "dw_available"
        ] = 1

        print(
            "\n✓ Dynamic World features added"
        )


# ================================================================
# 14. DISPLAY FINAL DW STATE
# ================================================================

print("\n" + "=" * 70)
print("FINAL DYNAMIC WORLD FEATURE STATE")
print("=" * 70)

for band in DW_FEATURES:

    value = feature_dict.get(
        band,
        None
    )

    print(
        f"{band:<28}: {value}"
    )


# ================================================================
# 15. CHECK MODEL SCHEMA
# ================================================================

print("\n" + "=" * 70)
print("74-FEATURE READINESS CHECK")
print("=" * 70)

if "ml_features" not in globals():

    raise RuntimeError(
        "ml_features not found. "
        "Run Cell 1."
    )

missing_model_features = [
    f
    for f in ml_features
    if f not in feature_dict
]

print(
    "Required model features:",
    len(ml_features)
)

print(
    "Current feature_dict:",
    len(feature_dict)
)

print(
    "Missing model features:",
    len(missing_model_features)
)

if missing_model_features:

    for f in missing_model_features:
        print("  ✗", f)

    raise RuntimeError(
        "Model feature dictionary is incomplete."
    )

print(
    "\n✓ All 74 model features exist."
)

print(
    "✓ Dynamic World missing values are "
    "handled training-compatibly."
)

print(
    "\nDO NOT run Random Forest directly."
)

print(
    "Next: Cell 2C will create the exact "
    "74-feature model vector and apply:"
)

print(
    "    fillna(train_medians).fillna(0)"
)


CELL 2B — LIVE DYNAMIC WORLD RETRIEVAL
✓ Cell 2A / Cell 2A-P variables found
Hotspot latitude : 23.805300
Hotspot longitude: 85.538310
Hotspot datetime : 2026-09-08 20:08:00+00:00

----------------------------------------------------------------------
CHECKING EARTH ENGINE PYTHON API
----------------------------------------------------------------------
✓ Earth Engine API already installed

----------------------------------------------------------------------
INITIALIZING GOOGLE EARTH ENGINE
----------------------------------------------------------------------
Earth Engine project: planar-depth-508020-q9
✓ Earth Engine initialized successfully
✓ Project ID: planar-depth-508020-q9

✓ Hotspot point created

----------------------------------------------------------------------
FIRMS TARGET
----------------------------------------------------------------------
FIRMS datetime: 2026-09-08 20:08:00+00:00
FIRMS date    : 2026-09-08

---------------------------------------------------------

In [61]:
# ================================================================
# CELL 2C — EXACT 74-FEATURE MODEL INPUT + PREDICTION
# ================================================================

import numpy as np
import pandas as pd

print("\n" + "=" * 70)
print("CELL 2C — FINAL 74-FEATURE MODEL ASSEMBLY")
print("=" * 70)


# ================================================================
# 1. REQUIRED TRAINING OBJECTS
# ================================================================

required_training_objects = [
    "final_model",
    "ml_features",
    "train_medians",
]

missing_training_objects = [
    x
    for x in required_training_objects
    if x not in globals()
]

if missing_training_objects:

    print("\nMissing training objects:")

    for x in missing_training_objects:
        print("  ✗", x)

    raise RuntimeError(
        "\nRequired training objects are missing. "
        "Run Cell 1 first."
    )

print("✓ final_model found")
print("✓ ml_features found")
print("✓ train_medians found")


# ================================================================
# 2. VERIFY MODEL FEATURE COUNT
# ================================================================

print("\n" + "-" * 70)
print("MODEL FEATURE SCHEMA")
print("-" * 70)

print(
    "Expected model features:",
    len(ml_features)
)

if len(ml_features) != 74:

    raise RuntimeError(
        f"Expected exactly 74 model features, "
        f"but ml_features contains {len(ml_features)}."
    )

print("✓ Exact 74-feature schema confirmed")


# ================================================================
# 3. VERIFY CURRENT FEATURE DICTIONARY
# ================================================================

if "feature_dict" not in globals():

    raise RuntimeError(
        "feature_dict not found. "
        "Run Cell 2A, Cell 2A-P and Cell 2B first."
    )

print(
    "\nCurrent feature_dict features:",
    len(feature_dict)
)

missing_features = [
    f
    for f in ml_features
    if f not in feature_dict
]

if missing_features:

    print("\nMissing features:")

    for f in missing_features:
        print("  ✗", f)

    raise RuntimeError(
        "Cannot construct the 74-feature model input."
    )

print("✓ All 74 required features exist")


# ================================================================
# 4. CREATE RAW MODEL INPUT
# ================================================================
#
# IMPORTANT:
# Preserve EXACT training feature order.
# ================================================================

X_live_raw = pd.DataFrame(
    [
        [
            feature_dict[f]
            for f in ml_features
        ]
    ],
    columns=ml_features
)

print("\n✓ Raw model input created")
print(
    "Shape:",
    X_live_raw.shape
)


# ================================================================
# 5. DISPLAY MISSING VALUES BEFORE IMPUTATION
# ================================================================

nan_features = [
    f
    for f in ml_features
    if pd.isna(X_live_raw.loc[0, f])
]

inf_features = [
    f
    for f in ml_features
    if np.isinf(
        X_live_raw.loc[0, f]
    )
]

print("\n" + "-" * 70)
print("RAW INPUT VALIDATION")
print("-" * 70)

print(
    "NaN features:",
    len(nan_features)
)

if nan_features:

    for f in nan_features:
        print(
            f"  NaN → {f}"
        )

print(
    "Infinite features:",
    len(inf_features)
)

if inf_features:

    for f in inf_features:
        print(
            f"  Inf → {f}"
        )


# ================================================================
# 6. APPLY EXACT TRAINING IMPUTATION
# ================================================================
#
# Original training preprocessing:
#
# train_medians = X_train.median()
#
# X_train = X_train.fillna(train_medians).fillna(0)
# X_test  = X_test.fillna(train_medians).fillna(0)
#
# We reproduce the same logic here.
# ================================================================

print("\n" + "-" * 70)
print("APPLYING TRAINING-COMPATIBLE IMPUTATION")
print("-" * 70)

X_live = (
    X_live_raw
    .fillna(train_medians)
    .fillna(0)
)

print(
    "✓ train_medians applied"
)

print(
    "✓ Remaining NaN → 0 fallback applied"
)


# ================================================================
# 7. CHECK FOR REMAINING NaN / INF
# ================================================================

remaining_nan = int(
    X_live.isna().sum().sum()
)

remaining_inf = int(
    np.isinf(
        X_live.to_numpy(
            dtype=float
        )
    ).sum()
)

print("\nFinal input validation:")

print(
    "Remaining NaN:",
    remaining_nan
)

print(
    "Remaining Inf:",
    remaining_inf
)

if remaining_nan != 0:

    raise RuntimeError(
        "Final model input still contains NaN values."
    )

if remaining_inf != 0:

    raise RuntimeError(
        "Final model input still contains infinite values."
    )

print(
    "✓ Final model input contains no NaN/Inf"
)


# ================================================================
# 8. VERIFY FEATURE ORDER ONE MORE TIME
# ================================================================

print("\n" + "-" * 70)
print("EXACT FEATURE ORDER CHECK")
print("-" * 70)

for i, feature_name in enumerate(
    ml_features,
    start=1
):

    print(
        f"{i:02d}. "
        f"{feature_name:<35} = "
        f"{X_live.loc[0, feature_name]}"
    )

print(
    "\n✓ Feature order exactly matches ml_features"
)


# ================================================================
# 9. VERIFY MODEL EXPECTED FEATURE COUNT
# ================================================================

if hasattr(
    final_model,
    "n_features_in_"
):

    model_feature_count = (
        final_model.n_features_in_
    )

    print("\n" + "-" * 70)
    print("MODEL INPUT COMPATIBILITY")
    print("-" * 70)

    print(
        "Random Forest expects:",
        model_feature_count
    )

    print(
        "Live vector contains:",
        X_live.shape[1]
    )

    if model_feature_count != X_live.shape[1]:

        raise RuntimeError(
            "Model feature count does not match "
            "the live input feature count."
        )

    print(
        "✓ Model and input feature counts match"
    )


# ================================================================
# 10. PREDICTION
# ================================================================

print("\n" + "=" * 70)
print("RUNNING 2025-TRAINED RANDOM FOREST")
print("=" * 70)

prediction_numeric = (
    final_model
    .predict(X_live)[0]
)

print(
    "Predicted numeric class:",
    prediction_numeric
)


# ================================================================
# 11. CLASS MAPPING
# ================================================================

NUMBER_TO_CLASS = {
    0: "Other",
    1: "Industrial Fire",
    2: "Gas Flare",
    3: "Agricultural Fire",
    4: "Mining Activity",
    5: "Wildfire",
}

prediction_numeric_int = int(
    prediction_numeric
)

if prediction_numeric_int not in NUMBER_TO_CLASS:

    raise RuntimeError(
        f"Unknown predicted class: "
        f"{prediction_numeric_int}"
    )

predicted_class = NUMBER_TO_CLASS[
    prediction_numeric_int
]


# ================================================================
# 12. PREDICTION PROBABILITIES
# ================================================================

probability_table = None

if hasattr(
    final_model,
    "predict_proba"
):

    probabilities = (
        final_model
        .predict_proba(X_live)[0]
    )

    classes = (
        final_model.classes_
    )

    probability_table = (
        pd.DataFrame({
            "class_number":
                classes.astype(int),

            "class_name": [
                NUMBER_TO_CLASS[
                    int(c)
                ]
                for c in classes
            ],

            "probability":
                probabilities
        })
        .sort_values(
            "probability",
            ascending=False
        )
        .reset_index(drop=True)
    )


# ================================================================
# 13. FINAL RESULT
# ================================================================

print("\n" + "=" * 70)
print("FINAL FIRE-SOURCE PREDICTION")
print("=" * 70)

print(
    "Hotspot latitude :",
    f"{live_hotspot_lat:.6f}"
)

print(
    "Hotspot longitude:",
    f"{live_hotspot_lon:.6f}"
)

print(
    "FIRMS datetime   :",
    live_hotspot_datetime
)

print(
    "\nPredicted class:",
    predicted_class
)

print(
    "Numeric class   :",
    prediction_numeric_int
)


# ================================================================
# 14. PROBABILITY TABLE
# ================================================================

if probability_table is not None:

    print("\n" + "-" * 70)
    print("CLASS PROBABILITIES")
    print("-" * 70)

    print(
        probability_table.to_string(
            index=False,
            formatters={
                "probability":
                    lambda x:
                    f"{x:.6f}"
            }
        )
    )


# ================================================================
# 15. TOP-CLASS CONFIDENCE
# ================================================================

if probability_table is not None:

    top_probability = float(
        probability_table
        .iloc[0]["probability"]
    )

    print("\n" + "-" * 70)
    print("TOP PREDICTION")
    print("-" * 70)

    print(
        "Class:",
        probability_table.iloc[0][
            "class_name"
        ]
    )

    print(
        "Probability:",
        f"{top_probability:.6f}"
    )


# ================================================================
# 16. SAVE FINAL LIVE VECTOR
# ================================================================

live_model_input = X_live.copy()

print("\n" + "=" * 70)
print("CELL 2C COMPLETED SUCCESSFULLY")
print("=" * 70)

print(
    "✓ Exact 74-feature vector created"
)

print(
    "✓ Training feature order preserved"
)

print(
    "✓ Training medians applied"
)

print(
    "✓ No NaN/Inf remains"
)

print(
    "✓ Random Forest prediction completed"
)

print(
    "\nIMPORTANT:"
)

print(
    "This is a PREDICTED SOURCE CLASS from "
    "the 2025-trained Random Forest."
)

print(
    "It is NOT a NASA FIRMS source label."
)


CELL 2C — FINAL 74-FEATURE MODEL ASSEMBLY
✓ final_model found
✓ ml_features found
✓ train_medians found

----------------------------------------------------------------------
MODEL FEATURE SCHEMA
----------------------------------------------------------------------
Expected model features: 74
✓ Exact 74-feature schema confirmed

Current feature_dict features: 74
✓ All 74 required features exist

✓ Raw model input created
Shape: (1, 74)

----------------------------------------------------------------------
RAW INPUT VALIDATION
----------------------------------------------------------------------
NaN features: 11
  NaN → water
  NaN → trees
  NaN → grass
  NaN → flooded_vegetation
  NaN → crops
  NaN → shrub_and_scrub
  NaN → built
  NaN → bare
  NaN → snow_and_ice
  NaN → dw_date_difference_days
  NaN → dw_max_probability
Infinite features: 0

----------------------------------------------------------------------
APPLYING TRAINING-COMPATIBLE IMPUTATION
-----------------------------